In [1]:
# CELL 1: Environment Setup & Windows Smart App Control DLL Bypass
import os
import re
import io
import gc
import sys
import glob
import json
import time
import math
import copy
import shutil
import random
import hashlib
import datetime
import py_compile
import subprocess
from collections import Counter, defaultdict
from types import SimpleNamespace

# Permanently block unused torchaudio/torchcodec imports (prevents WinError 4551)
sys.modules["torchaudio"] = None
sys.modules["torchcodec"] = None
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import open_clip
import nltk
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from pycocotools.coco import COCO
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from nltk.translate.meteor_score import meteor_score

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_all_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

print(f"PyTorch: {torch.__version__} | Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)")

PyTorch: 2.11.0+cu128 | Device: cuda
GPU    : NVIDIA GeForce RTX 4050 Laptop GPU (6.00 GB VRAM)


In [2]:
# CELL 2: RSICD-Parity Configuration Builder (100% Isolated Per Dataset)
def build_dataset_config(
    dataset_name: str,
    total_images: int = None,
    train_count: int = None,
    val_count: int = None,
    test_count: int = None,
):
    ds = dataset_name.lower().strip()
    assert ds in ("ucm", "nwpu"), "Supported datasets for this benchmark notebook: 'ucm', 'nwpu'"

    defaults = {
        "ucm":  {"folder": "vansh_ucm",  "total": 2100, "train": 1680, "val": 210, "test": 210, "max_len": 20},
        "nwpu": {"folder": "vansh_nwpu", "total": 4500, "train": 3600, "val": 450, "test": 450, "max_len": 25},
    }[ds]

    total = total_images or defaults["total"]
    tr = train_count or defaults["train"]
    va = val_count or defaults["val"]
    te = test_count or defaults["test"]
    assert tr + va + te == total, f"Split mismatch: {tr}+{va}+{te} != {total}"

    is_kaggle = os.path.exists("/kaggle")
    workspace_dir = f"/kaggle/working/{ds}" if is_kaggle else os.path.join(os.path.expanduser("~"), "Desktop", defaults["folder"])

    cfg = SimpleNamespace(
        ACTIVE_DATASET=ds,
        SEED=42,
        IS_KAGGLE=is_kaggle,
        WORKSPACE_DIR=workspace_dir,
        REPO_DIR=os.path.join(workspace_dir, "HATNet"),
        RAW_DATA_DIR=os.path.join(workspace_dir, "raw_data"),
        OUT_DATA_DIR=os.path.join(workspace_dir, "data"),
        FEATURES_DIR=os.path.join(workspace_dir, "features"),
        CHECKPOINT_DIR=os.path.join(workspace_dir, "checkpoints"),
        RESULTS_DIR=os.path.join(workspace_dir, "results"),

        # Dataset-prefixed independent JSON files
        MANIFEST_FILE=os.path.join(workspace_dir, f"{ds}_split_manifest.json"),
        CONFIG_JSON=os.path.join(workspace_dir, f"{ds}_configuration.json"),
        FEATURE_CACHE_META_JSON=os.path.join(workspace_dir, f"{ds}_feature_cache_metadata.json"),
        RETRIEVAL_META_JSON=os.path.join(workspace_dir, f"{ds}_retrieval_memory_metadata.json"),
        TRAINING_HISTORY_JSON=os.path.join(workspace_dir, f"{ds}_training_history.json"),
        VAL_PREDS_JSON=os.path.join(workspace_dir, f"{ds}_validation_predictions.json"),
        TEST_PREDS_JSON=os.path.join(workspace_dir, f"{ds}_test_predictions.json"),
        TEST_METRICS_JSON=os.path.join(workspace_dir, f"{ds}_test_metrics.json"),
        RUN_SUMMARY_TXT=os.path.join(workspace_dir, f"{ds}_run_summary.txt"),
        DATASET_JSON_OUT=os.path.join(workspace_dir, "data", f"dataset_{ds}_{total}.json"),

        # Split specs
        TOTAL_IMAGES=total,
        TRAIN_COUNT=tr,
        VAL_COUNT=va,
        TEST_COUNT=te,
        MAX_CAPTION_LEN=defaults["max_len"],
        MIN_WORD_FREQ=2,
        SEQ_PER_IMG=5,

        # GeoRSCLIP (identical to RSICD)
        MODEL_NAME="ViT-H-14",
        EXPECTED_CKPT_SHA256="62a3e79df886875976901b4a9b8d6a1c42b457879a5ce625c8eba9460cbc3a11",
        ALLOW_CKPT_HASH_OVERRIDE=True,
        EXTRACTION_LAYERS=[10, 21, 31],
        NUM_PATCHES=256,
        ATT_FEAT_SIZE=3840,
        CONCAT_ORDER="channels_10_21_31",
        CACHE_SCHEMA_VERSION="2.0.0",
        EXTRACT_BATCH_SIZE=16,

        # HATNet Architecture (identical to RSICD)
        CAPTION_MODEL="transformer",
        D_MODEL=512,
        D_FF=2048,
        NUM_HEADS=8,
        N_ENC=3,
        N_DEC=3,
        DROPOUT=0.1,
        DROP_PROB_LM=0.1,

        # Retrieval & ViPCap Hyperparameters (identical to RSICD)
        RETRIEVAL_TOP_K=6,
        RETRIEVAL_SIMILARITY="cosine",
        RETRIEVAL_TEMPERATURE=0.05,
        RETRIEVAL_SCORE_THRESHOLD=-1.0,
        RETRIEVAL_MEMORY_MODE="multiscale_b21_b31_mean_max",
        NUM_PROMPT_TOKENS=16,
        PROMPT_DROPOUT=0.1,

        # Training & Evaluation Hyperparameters (identical to RSICD)
        BATCH_SIZE=16,
        LEARNING_RATE=3e-4,
        WARMUP_EPOCHS=2,        # Dynamically scaled to 2 epochs (same ratio as RSICD)
        MAX_EPOCHS=18,          # Model A & B
        STAGE1_EPOCHS=3,        # Model C Stage 1 (frozen backbone)
        STAGE2_EPOCHS=7,        # Model C Stage 2 (joint fine-tuning)
        LABEL_SMOOTHING=0.05,
        GRAD_CLIP_NORM=0.1,
        BEAM_SIZE=3,
    )

    for d in [cfg.WORKSPACE_DIR, cfg.RAW_DATA_DIR, cfg.OUT_DATA_DIR,
              cfg.FEATURES_DIR, cfg.CHECKPOINT_DIR, cfg.RESULTS_DIR]:
        os.makedirs(d, exist_ok=True)

    with open(cfg.CONFIG_JSON, "w", encoding="utf-8") as f:
        json.dump(vars(cfg), f, indent=2)

    print(f"[{ds.upper()} CONFIG READY] Workspace: {cfg.WORKSPACE_DIR} | Split: {tr}/{va}/{te} (Total {total})")
    return cfg

In [3]:
# CELL 3: Universal Dataset Loader (Fast Columnar Export + Reuses UCM GeoRSCLIP Checkpoint + Live Flush)
def compute_file_sha256(filepath: str) -> str:
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

def normalize_caption_tokens(raw_text: str):
    cleaned = re.sub(r"[^a-z0-9\s]+", " ", str(raw_text).lower())
    return [t for t in cleaned.split() if t]

class LocalImageFolderWrapper:
    def __init__(self, path_list: list):
        self.path_list = path_list
    def __len__(self):
        return len(self.path_list)
    def __getitem__(self, idx):
        p = self.path_list[idx]
        return {"filename": os.path.basename(p), "image": Image.open(p).convert("RGB")}

class HFImageDatasetAdapter:
    """Normalizes any HuggingFace image dataset so adapter[idx]['image'] always returns an RGB PIL.Image."""
    def __init__(self, hf_ds, img_col: str, idx_to_fname: dict):
        self.hf_ds = hf_ds
        self.img_col = img_col
        self.idx_to_fname = idx_to_fname
    def __len__(self):
        return len(self.hf_ds)
    def __getitem__(self, idx):
        val = self.hf_ds[int(idx)][self.img_col]
        if isinstance(val, Image.Image):
            pil_img = val.convert("RGB")
        elif isinstance(val, dict) and val.get("bytes") is not None:
            pil_img = Image.open(io.BytesIO(val["bytes"])).convert("RGB")
        elif isinstance(val, dict) and val.get("path"):
            pil_img = Image.open(val["path"]).convert("RGB")
        else:
            pil_img = Image.open(io.BytesIO(val)).convert("RGB")
        return {"filename": self.idx_to_fname[int(idx)], "image": pil_img}

def extract_scene_class(record: dict, numeric_block: int = 100) -> str:
    for k in ("class", "category", "label", "scene", "class_name"):
        if k in record and record[k] is not None and str(record[k]).strip():
            return str(record[k]).lower().strip()
    base = os.path.basename(str(record["filename"])).lower()
    stem = os.path.splitext(base)[0]
    m_under = re.match(r"^([a-z_]+?)_?(\d+)$", stem)
    if m_under:
        prefix, num_str = m_under.group(1).strip("_"), m_under.group(2)
        if prefix and prefix not in ("ucm", "nwpu", "rsicd", "img", "image"):
            return prefix
        num = int(num_str)
        block = 700 if "nwpu" in prefix else numeric_block
        return f"class_{max(0, num - 1) // block:02d}"
    if stem.isdigit():
        return f"class_{max(0, int(stem) - 1) // numeric_block:02d}"
    return "unknown"

def _extract_row_captions(row: dict, cap_cols: list) -> list:
    caps = []
    for col in cap_cols:
        val = row.get(col)
        if val is None:
            continue
        if isinstance(val, list):
            for item in val:
                txt = item["raw"] if isinstance(item, dict) and "raw" in item else str(item)
                if txt.strip():
                    caps.append(txt.strip())
        elif isinstance(val, str) and val.strip():
            caps.append(val.strip())
    return caps

def discover_dataset_and_ckpt(cfg):
    ds = cfg.ACTIVE_DATASET
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")
    ignore_dirs = {"appdata", "anaconda3", "miniconda3", "site-packages", ".git", ".ipynb_checkpoints"}

    # 1. Read-only search for existing RS5M_ViT-H-14.pt (includes vansh_ucm so NWPU skips 3.94 GB download)
    ckpt_roots = [
        cfg.WORKSPACE_DIR,
        os.path.join(desktop, "vansh_ucm"),
        os.path.join(desktop, "vansh_2"),
        os.path.join(desktop, "vansh"),
        os.path.join(desktop, "vansh", "8k"),
    ]
    cand_ckpts = []
    direct_dl = os.path.join(os.path.expanduser("~"), "Downloads", "RS5M_ViT-H-14.pt")
    if os.path.exists(direct_dl):
        cand_ckpts.append(os.path.abspath(direct_dl))

    for root in list(dict.fromkeys(os.path.abspath(r) for r in ckpt_roots if os.path.exists(r))):
        root_depth = root.rstrip(os.path.sep).count(os.path.sep)
        for dp, dirs, fns in os.walk(root):
            dirs[:] = [d for d in dirs if d.lower() not in ignore_dirs and not d.startswith(".")]
            if dp.rstrip(os.path.sep).count(os.path.sep) - root_depth > 3:
                del dirs[:]
            for f in fns:
                if f == "RS5M_ViT-H-14.pt" or (f.endswith(".pt") and "ViT-H-14" in f):
                    cand_ckpts.append(os.path.abspath(os.path.join(dp, f)))
    cand_ckpts = list(dict.fromkeys(cand_ckpts))

    # 2. Search local dataset directories first
    ds_roots = [
        cfg.RAW_DATA_DIR,
        cfg.WORKSPACE_DIR,
        os.path.join(desktop, ds),
        os.path.join(desktop, f"{ds}_captions"),
        os.path.join(desktop, f"{ds.upper()}-Captions"),
    ]
    cand_jsons = []
    local_img_paths = {}
    local_stem_paths = {}
    img_exts = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")
    skip_json_tags = (
        "manifest", "metrics", "pred", "info.json", "configuration", "metadata",
        "history", "_train.json", "_val.json", "_test.json", f"dataset_{ds}_", f"raw_{ds}_exported"
    )

    for root in list(dict.fromkeys(os.path.abspath(r) for r in ds_roots if os.path.exists(r))):
        for dp, dirs, fns in os.walk(root):
            dirs[:] = [d for d in dirs if d.lower() not in ignore_dirs and not d.startswith(".")]
            dp_low = dp.lower()
            is_target_dir = (ds in dp_low) or (os.path.abspath(dp).startswith(os.path.abspath(cfg.RAW_DATA_DIR)))
            if not is_target_dir:
                continue
            for f in fns:
                low_f = f.lower()
                full_p = os.path.abspath(os.path.join(dp, f))
                if low_f.endswith(".json") and not any(t in low_f for t in skip_json_tags):
                    cand_jsons.append(full_p)
                elif low_f.endswith(img_exts):
                    local_img_paths[f] = full_p
                    local_stem_paths[os.path.splitext(low_f)[0]] = full_p

    cand_jsons = list(dict.fromkeys(cand_jsons))

    if cand_jsons and len(local_img_paths) >= cfg.TOTAL_IMAGES:
        print(f"[{ds.upper()} LOCAL FOUND] {len(local_img_paths)} images | JSON: {cand_jsons[0]}", flush=True)
        sorted_paths = sorted(local_img_paths.values())
        wrapper = LocalImageFolderWrapper(sorted_paths)
        fname_to_idx = {os.path.basename(p): i for i, p in enumerate(sorted_paths)}
        stem_to_idx = {os.path.splitext(os.path.basename(p).lower())[0]: i for i, p in enumerate(sorted_paths)}
        return cand_jsons[0], cand_ckpts, wrapper, fname_to_idx, stem_to_idx

    # 3. Verified HuggingFace Hub mirrors for UCM-Captions & NWPU-Captions
    hf_repos = {
        "ucm": ["caglarmert/UCM-captions"],
        "nwpu": ["caglarmert/NWPU-captions"],
    }[ds]

    hf_dataset, last_err = None, None
    for repo in hf_repos:
        try:
            from datasets import load_dataset, concatenate_datasets
            print(f"Downloading / loading HuggingFace dataset '{repo}' (NWPU is ~2.2 GB on first download)...", flush=True)
            raw = load_dataset(repo)
            hf_dataset = concatenate_datasets([raw[s] for s in raw.keys()]) if hasattr(raw, "keys") else raw
            print(f"[{ds.upper()} HF LOADED] {len(hf_dataset)} rows | Columns: {hf_dataset.column_names}", flush=True)
            break
        except Exception as e:
            last_err = e

    if hf_dataset is None:
        raise RuntimeError(
            f"\n[{ds.upper()} DATASET MISSING]\n"
            f"Local files not found AND HuggingFace load failed ({last_err}).\n"
            f"Manual option: Download {ds.upper()}-Captions (images + annotation JSON) and unzip into:\n"
            f"   {cfg.RAW_DATA_DIR}\n"
            f"Then re-run this cell."
        )

    cols = list(hf_dataset.column_names)
    img_col = next((c for c in cols if "Image" in type(hf_dataset.features[c]).__name__ or c.lower() in ("image", "img")), None)
    assert img_col is not None, f"No image column found in {cols}!"

    fn_col = next((c for c in cols if c.lower() in ("filename", "file_name", "img_id", "image_id", "id", "name")), None)
    cls_col = next((c for c in cols if c.lower() in ("class", "category", "label", "scene", "class_name")), None)

    cap_cols = [c for c in cols if c not in (img_col, fn_col, cls_col) and any(k in c.lower() for k in ("caption", "raw", "sent", "text", "desc"))]
    if not cap_cols:
        cap_cols = [c for c in cols if c not in (img_col, fn_col, cls_col) and c.lower() != "split"]
    assert len(cap_cols) > 0, f"Could not identify caption columns in {cols}!"

    # Fast columnar extraction (avoids 31,500 slow Arrow row lookups)
    text_only_ds = hf_dataset.remove_columns([img_col])
    cols_data = {c: text_only_ds[c] for c in text_only_ds.column_names}
    n_rows = len(text_only_ds)

    fname_to_idx, stem_to_idx, idx_to_fname = {}, {}, {}
    records = []
    for idx in range(n_rows):
        raw_fn_val = cols_data[fn_col][idx] if fn_col else None
        raw_fn = str(raw_fn_val).strip() if raw_fn_val is not None else f"{ds}_{idx:05d}.jpg"
        fn = os.path.basename(raw_fn)
        if not os.path.splitext(fn)[1]:
            fn = f"{fn}.jpg"
        if fn in fname_to_idx:
            fn = f"{os.path.splitext(fn)[0]}_{idx}.jpg"

        fname_to_idx[fn] = idx
        stem_to_idx[os.path.splitext(fn.lower())[0]] = idx
        idx_to_fname[idx] = fn

        row_caps = {c: cols_data[c][idx] for c in cap_cols}
        raw_caps = _extract_row_captions(row_caps, cap_cols)
        sents = []
        for s_idx, txt in enumerate(raw_caps):
            toks = normalize_caption_tokens(txt)
            if toks:
                sents.append({"raw": " ".join(toks), "tokens": toks, "sentid": s_idx})
        rec = {"filename": fn, "sentences": sents}
        if cls_col and cols_data[cls_col][idx] is not None:
            rec["class"] = str(cols_data[cls_col][idx])
        records.append(rec)

    exported_json = os.path.join(cfg.OUT_DATA_DIR, f"raw_{ds}_exported.json")
    with open(exported_json, "w", encoding="utf-8") as f:
        json.dump({"images": records, "dataset": ds}, f)
    print(f"[{ds.upper()} EXPORTED] {len(records)} records | Sample[0]: '{records[0]['sentences'][0]['raw'] if records[0]['sentences'] else ''}'", flush=True)

    adapter = HFImageDatasetAdapter(hf_dataset, img_col, idx_to_fname)
    return exported_json, cand_ckpts, adapter, fname_to_idx, stem_to_idx


def prepare_dataset_context(cfg) -> dict:
    """Builds split manifest, train-only vocab, IDF weights, H5 labels, and COCO references for one dataset."""
    resolved_json, cand_ckpts, hf_dataset, fname_to_idx, stem_to_idx = discover_dataset_and_ckpt(cfg)
    ann_fp = compute_file_sha256(resolved_json)

    with open(resolved_json, "r", encoding="utf-8") as f:
        raw_obj = json.load(f)

    flat_entries = []
    if isinstance(raw_obj, dict) and "images" in raw_obj:
        flat_entries = raw_obj["images"]
    elif isinstance(raw_obj, dict):
        for cls_key, items in raw_obj.items():
            if isinstance(items, list):
                for it in items:
                    if isinstance(it, dict):
                        it_copy = dict(it)
                        it_copy.setdefault("class", cls_key)
                        flat_entries.append(it_copy)
    elif isinstance(raw_obj, list):
        flat_entries = raw_obj

    usable_records = []
    seen_fnames = set()
    for entry in flat_entries:
        fname = entry.get("filename") or entry.get("file_name")
        if not fname:
            continue
        base_fn = os.path.basename(str(fname))
        stem = os.path.splitext(base_fn.lower())[0]
        if base_fn in seen_fnames:
            continue

        if base_fn in fname_to_idx:
            resolved_fn = base_fn
        elif stem in stem_to_idx:
            resolved_fn = hf_dataset[stem_to_idx[stem]]["filename"]
            fname_to_idx[resolved_fn] = stem_to_idx[stem]
        else:
            continue

        raw_list = []
        if "sentences" in entry and isinstance(entry["sentences"], list):
            for s in entry["sentences"]:
                raw_list.append(s.get("raw", s.get("caption", "")) if isinstance(s, dict) else str(s))
        else:
            for k in ["raw", "raw_1", "raw_2", "raw_3", "raw_4", "raw_5"]:
                if k in entry and entry[k]:
                    raw_list.append(str(entry[k]))

        sents = []
        for raw_c in raw_list:
            toks = normalize_caption_tokens(raw_c)
            if toks:
                sents.append({"raw": " ".join(toks), "tokens": toks, "sentid": len(sents)})
        if not sents:
            continue

        seen_fnames.add(base_fn)
        rec = {
            "filename": resolved_fn,
            "sentences": sents,
            "raw_captions": [c["raw"] for c in sents],
        }
        if "class" in entry and entry["class"]:
            rec["class"] = str(entry["class"])
        usable_records.append(rec)

    usable_records.sort(key=lambda r: r["filename"])
    for new_id, r in enumerate(usable_records):
        r["imgid"] = new_id
        for s in r["sentences"]:
            s["imgid"] = new_id

    assert len(usable_records) >= cfg.TOTAL_IMAGES, (
        f"Found only {len(usable_records)} matched images for {cfg.ACTIVE_DATASET.upper()}, need {cfg.TOTAL_IMAGES}."
    )
    dataset_fp = hashlib.sha256(
        "|".join(f"{r['imgid']}:{r['filename']}:{len(r['sentences'])}" for r in usable_records).encode("utf-8")
    ).hexdigest()

    num_block = 700 if cfg.ACTIVE_DATASET == "nwpu" else 100
    usable_by_id = {r["imgid"]: r for r in usable_records}
    manifest = None
    if os.path.exists(cfg.MANIFEST_FILE):
        try:
            with open(cfg.MANIFEST_FILE, "r", encoding="utf-8") as f:
                m = json.load(f)
            t_ids, v_ids, te_ids = set(m["train_ids"]), set(m["val_ids"]), set(m["test_ids"])
            if (m.get("dataset") == cfg.ACTIVE_DATASET and m.get("seed") == cfg.SEED and m.get("stratified") is True and
                m.get("dataset_fingerprint") == dataset_fp and len(t_ids) == cfg.TRAIN_COUNT and
                len(v_ids) == cfg.VAL_COUNT and len(te_ids) == cfg.TEST_COUNT and
                t_ids.isdisjoint(v_ids) and t_ids.isdisjoint(te_ids) and v_ids.isdisjoint(te_ids)):
                manifest = m
                print(f"[{cfg.ACTIVE_DATASET.upper()}] Reusing verified stratified split manifest.", flush=True)
        except Exception:
            manifest = None

    if manifest is None:
        rng = random.Random(cfg.SEED)
        by_cls_all = defaultdict(list)
        for r in usable_records:
            by_cls_all[extract_scene_class(r, numeric_block=num_block)].append(r)

        per_cls = cfg.TOTAL_IMAGES // max(1, len(by_cls_all))
        selected, leftovers = [], []
        for c in sorted(by_cls_all.keys()):
            items = list(by_cls_all[c])
            rng.shuffle(items)
            take = min(len(items), per_cls)
            selected.extend(items[:take])
            leftovers.extend(items[take:])
        if len(selected) < cfg.TOTAL_IMAGES:
            rng.shuffle(leftovers)
            selected.extend(leftovers[: cfg.TOTAL_IMAGES - len(selected)])
        selected = selected[:cfg.TOTAL_IMAGES]

        by_cls = defaultdict(list)
        for r in selected:
            by_cls[extract_scene_class(r, numeric_block=num_block)].append(r)

        train_recs, val_recs, test_recs = [], [], []
        val_r = cfg.VAL_COUNT / float(cfg.TOTAL_IMAGES)
        test_r = cfg.TEST_COUNT / float(cfg.TOTAL_IMAGES)
        for c in sorted(by_cls.keys()):
            items = by_cls[c]
            n_val = int(round(len(items) * val_r))
            n_test = int(round(len(items) * test_r))
            n_tr = len(items) - n_val - n_test
            train_recs.extend(items[:n_tr])
            val_recs.extend(items[n_tr : n_tr + n_val])
            test_recs.extend(items[n_tr + n_val : n_tr + n_val + n_test])

        while len(val_recs) > cfg.VAL_COUNT: train_recs.append(val_recs.pop())
        while len(test_recs) > cfg.TEST_COUNT: train_recs.append(test_recs.pop())
        while len(val_recs) < cfg.VAL_COUNT: val_recs.append(train_recs.pop())
        while len(test_recs) < cfg.TEST_COUNT: test_recs.append(train_recs.pop())

        rng.shuffle(train_recs); rng.shuffle(val_recs); rng.shuffle(test_recs)
        split_fp = hashlib.sha256(
            (f"{dataset_fp}:{ann_fp}:{cfg.SEED}:{cfg.ACTIVE_DATASET}:" +
             ",".join(str(r["imgid"]) for r in (train_recs + val_recs + test_recs))).encode("utf-8")
        ).hexdigest()

        manifest = {
            "dataset": cfg.ACTIVE_DATASET, "seed": cfg.SEED, "stratified": True,
            "dataset_fingerprint": dataset_fp, "annotation_fingerprint": ann_fp,
            "split_fingerprint": split_fp, "total_images": cfg.TOTAL_IMAGES,
            "train_count": len(train_recs), "val_count": len(val_recs), "test_count": len(test_recs),
            "train_ids": [r["imgid"] for r in train_recs],
            "val_ids": [r["imgid"] for r in val_recs],
            "test_ids": [r["imgid"] for r in test_recs],
            "train_captions": {str(r["imgid"]): r["raw_captions"] for r in train_recs},
            "val_captions": {str(r["imgid"]): r["raw_captions"] for r in val_recs},
            "test_captions": {str(r["imgid"]): r["raw_captions"] for r in test_recs},
        }
        with open(cfg.MANIFEST_FILE, "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2)
        print(f"[{cfg.ACTIVE_DATASET.upper()}] Saved stratified split manifest ({len(by_cls)} scene groups).", flush=True)

    selected_images = []
    for sp, id_list in [("train", manifest["train_ids"]), ("val", manifest["val_ids"]), ("test", manifest["test_ids"])]:
        for iid in id_list:
            rec = copy.deepcopy(usable_by_id[iid])
            rec["split"] = sp
            selected_images.append(rec)

    with open(cfg.DATASET_JSON_OUT, "w", encoding="utf-8") as f:
        json.dump({"images": selected_images}, f, indent=2)

    train_records = [img for img in selected_images if img["split"] == "train"]
    word_freq = Counter()
    for img in train_records:
        for sent in img["sentences"]:
            word_freq.update(sent["tokens"])

    vocab_words = sorted([w for w, c in word_freq.items() if c >= cfg.MIN_WORD_FREQ])
    if "UNK" not in vocab_words:
        vocab_words.append("UNK")
    vocab_words.sort()

    itow = {str(i + 1): w for i, w in enumerate(vocab_words)}
    wtoi = {w: i + 1 for i, w in enumerate(vocab_words)}
    unk_idx = wtoi["UNK"]
    vocab_size = len(vocab_words)

    stop_words = {"a", "an", "the", "of", "in", "on", "is", "are", "with", "and", "near", "by", "to", "there", "it", "s", "some", "many", "several"}
    vocab_idf = torch.ones(vocab_size + 1, dtype=torch.float32)
    vocab_idf[0] = 0.0
    total_docs = max(1, len(train_records) * 5)
    for w, idx in wtoi.items():
        val = math.log(1.0 + total_docs / float(word_freq.get(w, 1) + 1))
        if w in stop_words:
            val *= 0.25
        vocab_idf[idx] = val
    vocab_idf[1:] = vocab_idf[1:] / vocab_idf[1:].mean().clamp(min=1e-4)

    ds = cfg.ACTIVE_DATASET
    h5_label_path = os.path.join(cfg.OUT_DATA_DIR, f"{ds}_label.h5")
    info_json_path = os.path.join(cfg.OUT_DATA_DIR, f"{ds}_info.json")

    total_caps = sum(len(img["sentences"]) for img in selected_images)
    label_arrays = np.zeros((total_caps, cfg.MAX_CAPTION_LEN), dtype="int32")
    label_length = np.zeros(total_caps, dtype="int32")
    label_start_ix = np.zeros(len(selected_images), dtype="int32")
    label_end_ix = np.zeros(len(selected_images), dtype="int32")

    cursor = 0
    for i, img in enumerate(selected_images):
        label_start_ix[i] = cursor + 1
        for sent in img["sentences"]:
            toks = sent["tokens"][:cfg.MAX_CAPTION_LEN]
            label_length[cursor] = len(toks)
            for k, tok in enumerate(toks):
                label_arrays[cursor, k] = wtoi.get(tok, unk_idx)
            cursor += 1
        label_end_ix[i] = cursor

    need_write_h5 = True
    if os.path.exists(h5_label_path):
        try:
            with h5py.File(h5_label_path, "r") as f:
                if "labels" in f and np.array_equal(f["labels"][:], label_arrays) and np.array_equal(f["label_start_ix"][:], label_start_ix):
                    need_write_h5 = False
        except Exception:
            need_write_h5 = True

    if need_write_h5:
        if os.path.exists(h5_label_path):
            os.remove(h5_label_path)
        with h5py.File(h5_label_path, "w") as f:
            f.create_dataset("labels", data=label_arrays)
            f.create_dataset("label_start_ix", data=label_start_ix)
            f.create_dataset("label_end_ix", data=label_end_ix)
            f.create_dataset("label_length", data=label_length)

    with open(info_json_path, "w", encoding="utf-8") as f:
        json.dump({
            "ix_to_word": itow,
            "images": [{"id": int(img["imgid"]), "file_path": img["filename"], "split": img["split"]} for img in selected_images]
        }, f, indent=2)

    def _write_coco(sp_name):
        out_p = os.path.join(cfg.OUT_DATA_DIR, f"{ds}_{sp_name}.json")
        sub = [img for img in selected_images if img["split"] == sp_name]
        obj = {"images": [], "annotations": [], "type": "captions", "licenses": [], "info": {}}
        aid = 1
        for img in sub:
            obj["images"].append({"id": int(img["imgid"]), "file_name": img["filename"]})
            for s in img["sentences"]:
                obj["annotations"].append({"id": aid, "image_id": int(img["imgid"]), "caption": s["raw"]})
                aid += 1
        with open(out_p, "w", encoding="utf-8") as f:
            json.dump(obj, f)
        return out_p

    ctx = {
        "cfg": cfg,
        "dataset_fp": dataset_fp,
        "ann_fp": ann_fp,
        "manifest": manifest,
        "selected_images": selected_images,
        "train_records": train_records,
        "train_ids_set": set(manifest["train_ids"]),
        "val_ids_set": set(manifest["val_ids"]),
        "test_ids_set": set(manifest["test_ids"]),
        "itow": itow,
        "wtoi": wtoi,
        "unk_idx": unk_idx,
        "vocab_size": vocab_size,
        "vocab_idf": vocab_idf,
        "h5_label_path": h5_label_path,
        "info_json_path": info_json_path,
        "train_coco_json": _write_coco("train"),
        "val_coco_json": _write_coco("val"),
        "test_coco_json": _write_coco("test"),
        "hf_dataset": hf_dataset,
        "fname_to_idx": fname_to_idx,
        "cand_ckpts": cand_ckpts,
    }
    print(f"[{ds.upper()} READY] Train/Val/Test = {len(ctx['train_ids_set'])}/{len(ctx['val_ids_set'])}/{len(ctx['test_ids_set'])} | Vocab = {vocab_size}", flush=True)
    return ctx

print("[CELL 3 READY] Fast dataset loader initialized.", flush=True)

[CELL 3 READY] Fast dataset loader initialized.


In [4]:
# CELL 4: Isolated HATNet Patcher + GeoRSCLIP Feature Extractor (Reuses Existing ViT-H-14 Checkpoint Read-Only)
def setup_isolated_hatnet_repo(cfg):
    repo_captioning = os.path.join(cfg.REPO_DIR, "captioning")
    if not os.path.exists(repo_captioning):
        print(f"Cloning HATNet into isolated folder: {cfg.REPO_DIR}...")
        subprocess.run(["git", "clone", "https://github.com/hanhan68/HATNet.git", cfg.REPO_DIR], check=True)

    # Clear cached captioning modules and prioritize this dataset's isolated HATNet repo
    for mod in list(sys.modules.keys()):
        if mod == "captioning" or mod.startswith("captioning."):
            del sys.modules[mod]
    if cfg.REPO_DIR in sys.path:
        sys.path.remove(cfg.REPO_DIR)
    sys.path.insert(0, cfg.REPO_DIR)

    # 1. Lazy models/__init__.py
    models_init = os.path.join(cfg.REPO_DIR, "captioning", "models", "__init__.py")
    with open(models_init, "w", encoding="utf-8") as f:
        f.write("def setup(opt):\n    if opt.caption_model == 'transformer':\n        from .GeoRSCLIP_mul_HAT_PTE import TransformerModel\n        return TransformerModel(opt)\n    raise Exception(f'Unsupported: {opt.caption_model}')\n")
    py_compile.compile(models_init, doraise=True)

    t_bridge = os.path.join(cfg.REPO_DIR, "captioning", "models", "TransformerModel.py")
    if not os.path.exists(t_bridge):
        with open(t_bridge, "w", encoding="utf-8") as f:
            f.write("from .GeoRSCLIP_mul_HAT_PTE import subsequent_mask, TransformerModel, Generator\n")

    # 2. Patch GeoRSCLIP_mul_HAT_PTE.py (Strip unused BertModel import + MultiLevelAttEmbed + DecoderLayer2 fix)
    pte_file = os.path.join(cfg.REPO_DIR, "captioning", "models", "GeoRSCLIP_mul_HAT_PTE.py")
    with open(pte_file, "r", encoding="utf-8") as f:
        pte_src = f.read().replace("\r\n", "\n")

    pte_src = re.sub(r"^\s*from\s+transformers\s+import\s+BertModel.*$", "# removed unused BertModel import", pte_src, flags=re.MULTILINE)

    multi_level_cls = '''
class MultiLevelAttEmbed(nn.Module):
    def __init__(self, att_feat_size, d_model, use_bn=0, drop_prob_lm=0.1):
        super(MultiLevelAttEmbed, self).__init__()
        self.d_model = d_model
        chunk_in = att_feat_size // 3 if att_feat_size > d_model * 3 else d_model
        def make_proj():
            layers = []
            if use_bn: layers.append(nn.BatchNorm1d(chunk_in))
            layers.extend([nn.Linear(chunk_in, d_model), nn.ReLU()])
            if drop_prob_lm > 0: layers.append(nn.Dropout(drop_prob_lm))
            if use_bn == 2: layers.append(nn.BatchNorm1d(d_model))
            return nn.Sequential(*layers)
        self.proj1, self.proj2, self.proj3 = make_proj(), make_proj(), make_proj()
    def __getitem__(self, idx): return [self.proj1, self.proj2, self.proj3][idx]
    def __len__(self): return 3
    def forward(self, x):
        c = x.shape[-1] // 3
        return torch.cat([self.proj1(x[..., 0*c:1*c]), self.proj2(x[..., 1*c:2*c]), self.proj3(x[..., 2*c:3*c])], dim=-1)
'''
    if "class MultiLevelAttEmbed" not in pte_src:
        pte_src = pte_src.replace("class EncoderDecoder(nn.Module):", multi_level_cls + "\nclass EncoderDecoder(nn.Module):", 1)
    if "self.yita = nn.Parameter" not in pte_src:
        pte_src, n_init = re.subn(
            r"(self\.fc_alpha2\s*=\s*nn\.Linear\(self\.d_model\s*\+\s*self\.d_model,\s*self\.d_model\))",
            r"\1\n        self.channel = ChannelAttention(self.d_model, self.d_model)\n        self.spatial = SpatialAttention()\n        self.yita = nn.Parameter(torch.tensor(0.2))",
            pte_src, count=1
        )
        assert n_init == 1
    if "self.channel(Vm)" not in pte_src:
        fwd_pat = re.compile(
            r"channel\s*=\s*ChannelAttention\([^)]*\)\.to\([^)]*\)\s*\n"
            r"\s*spatial\s*=\s*SpatialAttention\(\)\.to\([^)]*\)\s*\n"
            r"\s*yita\s*=\s*nn\.Parameter\(.*?\)\.to\([^)]*\)\s*\n"
            r"\s*Vm\s*=\s*self\.mul_level_ffn\(m\)\s*\*\s*yita\s*\+\s*memory\[\.\.\.,\s*512\s*\*\s*2:512\s*\*\s*3\]\s*\n"
            r"\s*Vm\s*=\s*Vm\.view\(Vm\.shape\[0\],\s*Vm\.shape\[2\],\s*16,\s*16\).*?\n"
            r"\s*Vc\s*=\s*channel\(Vm\)\.view\(Vm\.shape\[0\],\s*-1,\s*Vm\.shape\[1\]\)\s*\n"
            r"\s*Vs\s*=\s*spatial\(Vm\)\.view\(Vm\.shape\[0\],\s*-1,\s*Vm\.shape\[1\]\)"
        )
        fwd_rep = (
            "Vm = self.mul_level_ffn(m) * self.yita + memory[..., self.d_model * 2:self.d_model * 3]\n"
            "        Vm = Vm.view(Vm.shape[0], Vm.shape[2], 16, 16)\n"
            "        Vc = self.channel(Vm).view(Vm.shape[0], -1, Vm.shape[1])\n"
            "        Vs = self.spatial(Vm).view(Vm.shape[0], -1, Vm.shape[1])"
        )
        pte_src, n_fwd = fwd_pat.subn(fwd_rep, pte_src, count=1)
        assert n_fwd == 1
    if "MultiLevelAttEmbed(self.att_feat_size" not in pte_src:
        att_pat = re.compile(r"self\.att_embed\s*=\s*clones\([^)]*nn\.Sequential\(.*?\)\s*\)?,\s*3\)", re.DOTALL)
        pte_src, n_att = att_pat.subn("self.att_embed = MultiLevelAttEmbed(self.att_feat_size, self.d_model, self.use_bn, self.drop_prob_lm)", pte_src, count=1)
        assert n_att == 1
    if "nn.Identity()," not in pte_src:
        pte_src = pte_src.replace("nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Dropout(dropout)),", "nn.Identity(),")
    with open(pte_file, "w", encoding="utf-8") as f:
        f.write(pte_src)
    py_compile.compile(pte_file, doraise=True)

    # 3. Patch dataloader.py (Windows num_workers=0 + safe h5py)
    dl_file = os.path.join(cfg.REPO_DIR, "captioning", "data", "dataloader.py")
    with open(dl_file, "r", encoding="utf-8") as f:
        dl_src = f.read().replace("\r\n", "\n")
    dl_src = re.sub(r"self\.h5_label_file\s*=\s*h5py\.File\(self\.opt\.input_label_h5,\s*'r'(?:,\s*driver='core')?\)", "self.h5_label_file = h5py.File(self.opt.input_label_h5, 'r')", dl_src)
    if "if not self.opt.input_fc_dir:" not in dl_src:
        dl_src = dl_src.replace(
            "return np.float32(self.fc_loader.get(str(self.info['images'][ix]['id'])))",
            "return np.zeros((1,), dtype='float32') if not self.opt.input_fc_dir else np.float32(self.fc_loader.get(str(self.info['images'][ix]['id'])))"
        )
    dl_src = re.sub(
        r"num_workers\s*=\s*4[^\n]*\n\s*collate_fn\s*=\s*partial\([^)]*\)\s*,\s*\n\s*drop_last\s*=\s*False\s*,\s*\n\s*persistent_workers\s*=\s*True",
        "num_workers=getattr(self.opt, 'loader_num_workers', 0),\n                                                  collate_fn=partial(self.dataset.collate_func, split=split),\n                                                  drop_last=False,\n                                                  persistent_workers=(getattr(self.opt, 'loader_num_workers', 0) > 0)",
        dl_src
    )
    with open(dl_file, "w", encoding="utf-8") as f:
        f.write(dl_src)
    py_compile.compile(dl_file, doraise=True)

    rew_file = os.path.join(cfg.REPO_DIR, "captioning", "utils", "rewards.py")
    if os.path.exists(rew_file):
        with open(rew_file, "r", encoding="utf-8") as f:
            rs = f.read()
        if "print('cider or coco-caption missing')" in rs:
            with open(rew_file, "w", encoding="utf-8") as f:
                f.write(rs.replace("print('cider or coco-caption missing')", "pass"))

def extract_and_verify_features(ctx):
    cfg = ctx["cfg"]
    setup_isolated_hatnet_repo(cfg)

    cand_ckpts = ctx["cand_ckpts"]
    if not cand_ckpts:
        dl_p = hf_hub_download(repo_id="Zilun/GeoRSCLIP", filename="ckpt/RS5M_ViT-H-14.pt", local_dir=cfg.WORKSPACE_DIR, local_dir_use_symlinks=False)
        cand_ckpts = [os.path.abspath(dl_p)]
    ckpt_path = cand_ckpts[0]
    ckpt_sha = compute_file_sha256(ckpt_path)
    print(f"[{cfg.ACTIVE_DATASET.upper()}] Using GeoRSCLIP checkpoint: {ckpt_path}")

    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-H-14", pretrained=ckpt_path)
    clip_model = clip_model.to(DEVICE).eval()
    for p in clip_model.parameters():
        p.requires_grad = False
    prep_str = str(clip_preprocess)

    to_extract, cached_ok = [], 0
    for m in ctx["selected_images"]:
        iid = int(m["imgid"])
        fp = os.path.join(cfg.FEATURES_DIR, f"{iid}.npz")
        valid = False
        if os.path.exists(fp) and not os.path.exists(fp + ".tmp.npz"):
            try:
                with np.load(fp, allow_pickle=False) as d:
                    if (int(d["img_id"]) == iid and str(d["dataset_fingerprint"]) == ctx["dataset_fp"] and
                        d["feat"].shape == (cfg.NUM_PATCHES, cfg.ATT_FEAT_SIZE) and np.isfinite(d["feat"]).all()):
                        valid = True
            except Exception:
                valid = False
        if valid:
            cached_ok += 1
        else:
            if os.path.exists(fp + ".tmp.npz"):
                os.remove(fp + ".tmp.npz")
            to_extract.append(m)

    print(f"[{cfg.ACTIVE_DATASET.upper()} CACHE] {cached_ok} valid cached | {len(to_extract)} to extract.")
    target_layers = set(cfg.EXTRACTION_LAYERS)
    bs = cfg.EXTRACT_BATCH_SIZE

    for s_idx in tqdm(range(0, len(to_extract), bs), desc=f"Extracting {cfg.ACTIVE_DATASET.upper()} GeoRSCLIP"):
        batch_metas = to_extract[s_idx : s_idx + bs]
        pil_imgs = [clip_preprocess(ctx["hf_dataset"][ctx["fname_to_idx"][m["filename"]]]["image"].convert("RGB")) for m in batch_metas]
        batch_t = torch.stack(pil_imgs, dim=0).to(DEVICE)

        with torch.no_grad():
            x = clip_model.visual.conv1(batch_t)
            x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
            cls_tok = clip_model.visual.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)
            x = torch.cat([cls_tok, x], dim=1)
            x = clip_model.visual.ln_pre(x + clip_model.visual.positional_embedding.to(x.dtype))
            bf = getattr(clip_model.visual.transformer, "batch_first", True)
            if not bf:
                x = x.transpose(0, 1).contiguous()
            outs = []
            for blk_i, blk in enumerate(clip_model.visual.transformer.resblocks):
                x = blk(x)
                if blk_i in target_layers:
                    outs.append(x[:, 1:, :].clone() if bf else x[1:, :, :].transpose(0, 1).clone())
            concat_np = torch.cat(outs, dim=-1).cpu().numpy().astype(np.float32)

        for bi, m in enumerate(batch_metas):
            iid = int(m["imgid"])
            fp = os.path.join(cfg.FEATURES_DIR, f"{iid}.npz")
            tmp = fp + ".tmp.npz"
            np.savez_compressed(
                tmp, feat=concat_np[bi], img_id=np.int64(iid), filename=np.str_(m["filename"]),
                dataset_fingerprint=np.str_(ctx["dataset_fp"]), annotation_fingerprint=np.str_(ctx["ann_fp"]),
                checkpoint_sha256=np.str_(ckpt_sha), model_name=np.str_(cfg.MODEL_NAME),
                layers=np.array(cfg.EXTRACTION_LAYERS, dtype=np.int32), preprocessing=np.str_(prep_str),
                concat_order=np.str_(cfg.CONCAT_ORDER), shape=np.array(concat_np[bi].shape, dtype=np.int32),
                dtype=np.str_("float32"), cache_schema_version=np.str_(cfg.CACHE_SCHEMA_VERSION)
            )
            os.replace(tmp, fp)

    del clip_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    with open(cfg.FEATURE_CACHE_META_JSON, "w", encoding="utf-8") as f:
        json.dump({"dataset": cfg.ACTIVE_DATASET, "total_verified_files": len(ctx["selected_images"]),
                   "checkpoint_sha256": ckpt_sha, "dataset_fingerprint": ctx["dataset_fp"]}, f, indent=2)
    print(f"[{cfg.ACTIVE_DATASET.upper()}] All {len(ctx['selected_images'])} feature files verified!")

In [5]:
# CELL 5: Multi-Scale Retrieval Memory, Visually-Grounded IDF-Weighted ViPCap Models A/B/C & Evaluator
class TrainingOnlyRetrievalMemory:
    def __init__(self, ctx: dict):
        cfg = ctx["cfg"]
        self.val_ids_set = ctx["val_ids_set"]
        self.test_ids_set = ctx["test_ids_set"]
        self.features_dir = cfg.FEATURES_DIR
        self.default_top_k = cfg.RETRIEVAL_TOP_K
        self.default_temp = cfg.RETRIEVAL_TEMPERATURE
        self.default_thresh = cfg.RETRIEVAL_SCORE_THRESHOLD
        self.memory_mode = cfg.RETRIEVAL_MEMORY_MODE

        self.train_ids, self.id_to_index, self.captions_by_id = [], {}, {}
        tok_ids_list, tok_msks_list, embs = [], [], []

        for idx, rec in enumerate(ctx["train_records"]):
            iid = int(rec["imgid"])
            assert iid not in self.val_ids_set and iid not in self.test_ids_set
            self.train_ids.append(iid)
            self.id_to_index[iid] = idx
            self.captions_by_id[iid] = list(rec["raw_captions"])

            c_tok = np.zeros((5, cfg.MAX_CAPTION_LEN), dtype=np.int64)
            c_msk = np.zeros((5, cfg.MAX_CAPTION_LEN), dtype=np.float32)
            sents = rec["sentences"][:5]
            for ci in range(5):
                toks = sents[ci % len(sents)]["tokens"][:cfg.MAX_CAPTION_LEN]
                for ti, tok in enumerate(toks):
                    c_tok[ci, ti] = ctx["wtoi"].get(tok, ctx["unk_idx"])
                    c_msk[ci, ti] = 1.0
            tok_ids_list.append(c_tok)
            tok_msks_list.append(c_msk)

            with np.load(os.path.join(self.features_dir, f"{iid}.npz")) as d:
                feat = d["feat"]
            embs.append(self.compute_query_embedding_np(feat[None, ...])[0])

        self.train_ids_set = set(self.train_ids)
        self.memory_embeddings = torch.from_numpy(np.stack(embs, axis=0)).float()
        self.memory_token_ids = torch.from_numpy(np.stack(tok_ids_list, axis=0)).long()
        self.memory_token_masks = torch.from_numpy(np.stack(tok_msks_list, axis=0)).float()
        self.metadata = {
            "dataset": cfg.ACTIVE_DATASET, "memory_size": len(self.train_ids),
            "memory_mode": self.memory_mode, "top_k": self.default_top_k,
            "temperature": self.default_temp, "contains_val_or_test": False
        }
        with open(cfg.RETRIEVAL_META_JSON, "w", encoding="utf-8") as f:
            json.dump(self.metadata, f, indent=2)

    @staticmethod
    def _l2_np(x):
        return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-8)

    def compute_query_embedding_np(self, att_np):
        b21, b31 = att_np[:, :, 1280:2560], att_np[:, :, 2560:3840]
        comb = 0.60 * self._l2_np(b31.mean(axis=1)) + 0.25 * self._l2_np(b31.max(axis=1)) + 0.15 * self._l2_np(b21.mean(axis=1))
        return self._l2_np(comb).astype(np.float32)

    def compute_query_embedding_torch(self, att):
        b21, b31 = att[:, :, 1280:2560], att[:, :, 2560:3840]
        comb = 0.60 * F.normalize(b31.mean(dim=1), p=2, dim=-1) + 0.25 * F.normalize(b31.amax(dim=1), p=2, dim=-1) + 0.15 * F.normalize(b21.mean(dim=1), p=2, dim=-1)
        return F.normalize(comb, p=2, dim=-1)

    def retrieve_batch(self, query_att, query_image_ids=None, is_training=False, top_k=None):
        k = top_k or self.default_top_k
        device = query_att.device
        sim = torch.matmul(self.compute_query_embedding_torch(query_att), self.memory_embeddings.to(device).t())
        if is_training and query_image_ids is None:
            raise ValueError("query_image_ids required during training for strict self-exclusion!")
        if query_image_ids is not None:
            for bi, qid in enumerate(query_image_ids):
                idx = self.id_to_index.get(int(qid))
                if idx is not None:
                    sim[bi, idx] = -1e9
        top_scores, top_idx = torch.topk(sim, k=k, dim=-1, largest=True, sorted=True)
        sim_weights = F.softmax(top_scores / max(self.default_temp, 1e-4), dim=-1)
        cpu_idx = top_idx.detach().cpu()
        ret_ids = [[self.train_ids[j] for j in row.tolist()] for row in cpu_idx]
        return {
            "retrieved_ids": ret_ids, "top_scores": top_scores, "sim_weights": sim_weights,
            "token_ids": self.memory_token_ids[cpu_idx].to(device),
            "token_masks": self.memory_token_masks[cpu_idx].to(device),
        }


class RetrievedCaptionEncoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 512, dropout: float = 0.1, vocab_idf: torch.Tensor = None):
        super().__init__()
        self.d_model = d_model
        self.token_embed = nn.Embedding(vocab_size + 1, d_model, padding_idx=0)
        self.pos_embed = nn.Embedding(64, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.sent_proj = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.LayerNorm(d_model))
        idf_buf = vocab_idf.clone().float() if vocab_idf is not None else torch.ones(vocab_size + 1, dtype=torch.float32)
        idf_buf[0] = 0.0
        self.register_buffer("vocab_idf", idf_buf, persistent=True)  # Saved inside checkpoints!

    def forward(self, token_ids, token_masks, sim_weights):
        B, K, M, L = token_ids.shape
        pos_idx = torch.arange(L, device=token_ids.device).view(1, 1, 1, L)
        emb = self.dropout(self.norm(self.token_embed(token_ids) + self.pos_embed(pos_idx)))
        msk = token_masks.unsqueeze(-1)
        raw_cap = (emb * msk).sum(dim=3) / msk.sum(dim=3).clamp(min=1.0)
        per_cap = self.sent_proj(raw_cap)
        per_nbr = per_cap.mean(dim=2)
        w = sim_weights.unsqueeze(-1)
        weighted_summary = (per_nbr * w).sum(dim=1)

        cap_norm = F.normalize(raw_cap, p=2, dim=-1)
        mean_norm = F.normalize(raw_cap.mean(dim=2, keepdim=True), p=2, dim=-1)
        medoid_idx = (cap_norm * mean_norm).sum(dim=-1).argmax(dim=2)

        g_tok = medoid_idx.view(B, K, 1, 1, 1).expand(-1, -1, 1, L, self.d_model)
        g_id = medoid_idx.view(B, K, 1, 1).expand(-1, -1, 1, L)
        cons_emb = emb.gather(dim=2, index=g_tok).squeeze(2)
        cons_ids = token_ids.gather(dim=2, index=g_id).squeeze(2)
        cons_msk = token_masks.gather(dim=2, index=g_id).squeeze(2)

        idf_scale = self.vocab_idf[cons_ids.clamp(0, self.vocab_idf.shape[0] - 1)].unsqueeze(-1)
        cons_emb = cons_emb * (0.5 + 0.5 * idf_scale) * w.unsqueeze(-1)
        bank_tokens = torch.cat([per_cap * w.unsqueeze(-1), cons_emb], dim=2)
        bank_masks = torch.cat([torch.ones(B, K, M, device=token_ids.device, dtype=cons_msk.dtype), cons_msk], dim=2)
        return {
            "weighted_summary": weighted_summary,
            "token_bank": bank_tokens.view(B, K * (M + L), self.d_model),
            "bank_mask": bank_masks.view(B, K * (M + L)),
        }


class ViPCapPromptMapper(nn.Module):
    def __init__(self, d_model: int = 512, num_prompt_tokens: int = 16, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.num_prompt_tokens = num_prompt_tokens
        self.prompt_queries = nn.Parameter(torch.randn(1, num_prompt_tokens, d_model) * 0.02)
        self.vis_query_proj = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.LayerNorm(d_model))
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.summary_cond = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, enc_ret, vis_context=None):
        B = enc_ret["token_bank"].shape[0]
        queries = self.prompt_queries.expand(B, -1, -1)
        if vis_context is not None:
            queries = queries + self.vis_query_proj(vis_context).unsqueeze(1)
        attn_out, _ = self.cross_attn(query=queries, key=enc_ret["token_bank"], value=enc_ret["token_bank"], key_padding_mask=(enc_ret["bank_mask"] <= 0.0))
        p = self.norm1(queries + attn_out)
        s_exp = enc_ret["weighted_summary"].unsqueeze(1).expand(-1, self.num_prompt_tokens, -1)
        return self.norm2(p + self.summary_cond(torch.cat([p, s_exp], dim=-1)))


class VisualPromptFusion(nn.Module):
    def __init__(self, d_model: int = 512, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.visual_prompt_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.gate_proj = nn.Linear(d_model * 2, d_model)
        self.retrieval_only_proj = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.LayerNorm(d_model), nn.Tanh())
        self.fusion_alpha = nn.Parameter(torch.tensor(0.15))
        self.query_norm = nn.LayerNorm(d_model)
        self.prompt_norm = nn.LayerNorm(d_model)
        self.prompt_gate_proj = nn.Linear(d_model * 2, d_model)
        self.prompt_beta = nn.Parameter(torch.tensor(0.0))

    def forward(self, memory, prompt_tokens=None, retrieval_summary=None, ab_mode="full_vipcap"):
        if ab_mode == "baseline" or (prompt_tokens is None and retrieval_summary is None):
            return memory
        m1 = memory[..., 0*self.d_model : 1*self.d_model]
        m2 = memory[..., 1*self.d_model : 2*self.d_model]
        m3 = memory[..., 2*self.d_model : 3*self.d_model]
        if ab_mode == "retrieval_only":
            cond = self.retrieval_only_proj(retrieval_summary).unsqueeze(1).expand_as(m3)
            gate = torch.sigmoid(self.gate_proj(torch.cat([m3, cond], dim=-1)))
            m3_out = m3 + torch.tanh(self.fusion_alpha) * (gate * cond)
        elif ab_mode == "full_vipcap":
            cond = self.retrieval_only_proj(retrieval_summary).unsqueeze(1).expand_as(m3)
            gate = torch.sigmoid(self.gate_proj(torch.cat([m3, cond], dim=-1)))
            m3_base = m3 + torch.tanh(self.fusion_alpha) * (gate * cond)
            attn_delta, _ = self.visual_prompt_attn(query=self.query_norm(m3_base), key=prompt_tokens, value=prompt_tokens)
            attn_delta = self.prompt_norm(attn_delta)
            p_gate = torch.sigmoid(self.prompt_gate_proj(torch.cat([m3_base, attn_delta], dim=-1)))
            m3_out = m3_base + torch.tanh(self.prompt_beta) * (p_gate * attn_delta)
        else:
            raise ValueError(f"Unknown ab_mode: {ab_mode}")
        return torch.cat([m1, m2, m3_out], dim=-1)


class RetrievalEnhancedHATNet(nn.Module):
    def __init__(self, opt, use_retrieval=False, ab_mode="baseline", num_prompt_tokens=16, prompt_dropout=0.1, vocab_idf=None):
        super().__init__()
        from captioning.models import setup as setup_base_hatnet
        self.opt = copy.deepcopy(opt)
        self.use_retrieval = bool(use_retrieval and ab_mode != "baseline")
        self.ab_mode = "baseline" if not self.use_retrieval else ab_mode
        self.d_model = opt.d_model
        self.hatnet = setup_base_hatnet(self.opt)

        if self.use_retrieval:
            self.caption_encoder = RetrievedCaptionEncoder(self.opt.vocab_size, self.d_model, prompt_dropout, vocab_idf=vocab_idf)
            self.prompt_mapper = ViPCapPromptMapper(self.d_model, num_prompt_tokens, self.opt.num_att_heads, prompt_dropout) if self.ab_mode == "full_vipcap" else None
            self.prompt_fusion = VisualPromptFusion(self.d_model, self.opt.num_att_heads, prompt_dropout)
        else:
            self.caption_encoder = self.prompt_mapper = self.prompt_fusion = None

        self._active_ret = None
        orig_encode = self.hatnet.model.encode

        def patched_encode(src, src_mask):
            memory = orig_encode(src, src_mask)
            if not self.use_retrieval or self._active_ret is None:
                return memory
            enc_ret = self._active_ret
            ret_sum = enc_ret["weighted_summary"]
            B_mem, B_ret = memory.shape[0], ret_sum.shape[0]
            m3_all = memory[..., 2*self.d_model : 3*self.d_model]
            m3_unique = m3_all[0::(B_mem // B_ret)] if B_mem != B_ret else m3_all
            prompt_toks = self.prompt_mapper(enc_ret, vis_context=m3_unique.mean(dim=1)) if self.prompt_mapper is not None else None
            if prompt_toks is not None and prompt_toks.shape[0] != B_mem:
                rf = B_mem // prompt_toks.shape[0]
                prompt_toks = prompt_toks.unsqueeze(1).repeat(1, rf, 1, 1).view(B_mem, -1, self.d_model)
            if ret_sum.shape[0] != B_mem:
                rf = B_mem // ret_sum.shape[0]
                ret_sum = ret_sum.unsqueeze(1).repeat(1, rf, 1).view(B_mem, self.d_model)
            return self.prompt_fusion(memory, prompt_tokens=prompt_toks, retrieval_summary=ret_sum, ab_mode=self.ab_mode)

        self.hatnet.model.encode = patched_encode

    def forward(self, fc_feats, att_feats, seq=None, att_masks=None, retrieval_batch=None, opt=None, mode="forward"):
        if self.use_retrieval:
            self._active_ret = self.caption_encoder(retrieval_batch["token_ids"], retrieval_batch["token_masks"], retrieval_batch["sim_weights"])
        else:
            if retrieval_batch is not None:
                raise RuntimeError("Baseline model must not receive retrieval_batch!")
            self._active_ret = None
        try:
            if mode == "forward":
                return self.hatnet(fc_feats, att_feats, seq, att_masks)
            return self.hatnet(fc_feats, att_feats, att_masks, opt=opt or {}, mode="sample")
        finally:
            self._active_ret = None


class ConsistentCaptionEvaluator:
    VERSION = "1.0.0-fixed-karpathy-coco"
    DANGLING_TAIL = {"and", "or", "near", "with", "in", "on", "of", "to", "by", "at", "from", "a", "an", "the"}

    @staticmethod
    def decode_tokens(vocab: dict, seq: torch.Tensor) -> list:
        captions = []
        for i in range(seq.size(0)):
            raw_words = []
            for j in range(seq.size(1)):
                ix = int(seq[i, j].item())
                if ix > 0:
                    w = vocab.get(str(ix), vocab.get(ix, ""))
                    if w and w != "UNK":
                        raw_words.append(w)
                else:
                    break
            words = []
            for w in raw_words:
                if words and words[-1] == w: continue
                words.append(w)
                if len(words) >= 4 and words[-4:-2] == words[-2:]: del words[-2:]
                if len(words) >= 6 and words[-6:-3] == words[-3:]: del words[-3:]
            if len(words) >= 5 and words[-3] == "and" and words[-5:-3] == words[-2:]: del words[-3:]
            if len(words) >= 3 and words[-2] == "and" and words[-3] == words[-1]: del words[-2:]
            while len(words) > 2 and words[-1] in ConsistentCaptionEvaluator.DANGLING_TAIL:
                words.pop()
            captions.append(" ".join(words) if words else "aerial view")
        return captions

    @classmethod
    def evaluate_predictions(cls, coco_gt_file: str, predictions: list, expected_ids: set, try_spice: bool = False) -> dict:
        expected_ids_set = set(int(x) for x in expected_ids)
        assert set(int(p["image_id"]) for p in predictions) == expected_ids_set, "Prediction IDs mismatch GT!"
        coco_gt = COCO(coco_gt_file)
        pred_map = {int(p["image_id"]): " ".join(normalize_caption_tokens(p["caption"])) for p in predictions}
        gts, res = {}, {}
        for iid in sorted(expected_ids_set):
            gts[iid] = [" ".join(normalize_caption_tokens(a["caption"])) for a in coco_gt.imgToAnns[iid]]
            res[iid] = [pred_map[iid] or "aerial view"]
        metrics = {}
        bleu_scores, _ = Bleu(4).compute_score(gts, res)
        for n, v in zip(["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"], bleu_scores):
            metrics[n] = float(v)
        metrics["METEOR"] = float(np.mean([meteor_score([r.split() for r in gts[iid]], res[iid][0].split()) for iid in sorted(expected_ids_set)]))
        metrics["ROUGE_L"] = float(Rouge().compute_score(gts, res)[0])
        metrics["CIDEr"] = float(Cider().compute_score(gts, res)[0])
        if try_spice:
            try:
                from pycocoevalcap.spice.spice import Spice
                metrics["SPICE"] = float(Spice().compute_score(gts, res)[0])
            except Exception as e:
                metrics["SPICE"] = None
                metrics["SPICE_error"] = f"{type(e).__name__}: {e}"
        return metrics

print("[CELL 5 READY] Architecture, Retrieval Memory, and Evaluator loaded.")

[CELL 5 READY] Architecture, Retrieval Memory, and Evaluator loaded.


In [6]:
# CELL 6: Complete Training & Independent Test Evaluation Engine (Debugged for Windows Handles + Atomic Checkpoints)
class MaskedLabelSmoothingLoss(nn.Module):
    def __init__(self, smoothing: float = 0.05):
        super().__init__()
        self.smoothing = float(smoothing)

    def forward(self, log_probs, target, mask):
        if target.ndim == 3:
            target = target.reshape(-1, target.shape[-1])
        if mask.ndim == 3:
            mask = mask.reshape(-1, mask.shape[-1])
        L = min(log_probs.shape[1], target.shape[1], mask.shape[1])
        V = log_probs.shape[-1]
        log_probs = log_probs[:, :L, :].contiguous()
        target = target[:, :L].contiguous().clamp(0, V - 1)
        mask = mask[:, :L].contiguous().float()
        nll = -log_probs.gather(dim=-1, index=target.unsqueeze(-1)).squeeze(-1)
        smooth = -log_probs[:, :, 1:].mean(dim=-1) if V > 1 else -log_probs.mean(dim=-1)
        return (((1.0 - self.smoothing) * nll + self.smoothing * smooth) * mask).sum() / mask.sum().clamp(min=1.0)


def build_opt_from_ctx(ctx):
    cfg = ctx["cfg"]
    return SimpleNamespace(
        data_path=cfg.OUT_DATA_DIR, input_json=ctx["info_json_path"], input_label_h5=ctx["h5_label_path"],
        input_att_dir=cfg.FEATURES_DIR, input_fc_dir="", input_box_dir="",
        batch_size=cfg.BATCH_SIZE, seq_per_img=cfg.SEQ_PER_IMG, max_length=cfg.MAX_CAPTION_LEN,
        caption_model=cfg.CAPTION_MODEL, d_model=cfg.D_MODEL, d_ff=cfg.D_FF,
        num_att_heads=cfg.NUM_HEADS, num_layers=cfg.N_ENC, N_enc=cfg.N_ENC, N_dec=cfg.N_DEC,
        dropout=cfg.DROPOUT, input_encoding_size=cfg.D_MODEL, rnn_size=cfg.D_FF, drop_prob_lm=cfg.DROP_PROB_LM,
        use_bn=0, seq_length=cfg.MAX_CAPTION_LEN, eos_idx=0, pad_idx=0, fc_feat_size=0,
        att_feat_size=cfg.ATT_FEAT_SIZE, att_hid_size=cfg.D_MODEL, learning_rate=cfg.LEARNING_RATE,
        use_box=0, use_fc=False, use_att=True, norm_att_feat=0, loader_num_workers=0,
        vocab_size=ctx["vocab_size"], vocab=ctx["itow"],
    )


def _close_dataloader_h5(loader):
    try:
        if hasattr(loader, "dataset") and hasattr(loader.dataset, "h5_label_file"):
            loader.dataset.h5_label_file.close()
    except Exception:
        pass


def _atomic_torch_save(payload, path):
    tmp = path + ".tmp"
    torch.save(payload, tmp)
    os.replace(tmp, path)


@torch.no_grad()
def generate_split_predictions(model, split_ids, ret_mem, features_dir, top_k=6, beam_size=3, eval_bs=32, desc="Decoding"):
    model.eval()
    preds = []
    for i in tqdm(range(0, len(split_ids), eval_bs), desc=desc, leave=False):
        b_ids = [int(x) for x in split_ids[i : i + eval_bs]]
        feats = []
        for iid in b_ids:
            with np.load(os.path.join(features_dir, f"{iid}.npz")) as d:
                feats.append(d["feat"])
        att = torch.from_numpy(np.stack(feats, axis=0)).to(DEVICE)
        fc = torch.zeros(att.shape[0], 0, device=DEVICE)
        ret_b = None
        if model.use_retrieval:
            ret_b = ret_mem.retrieve_batch(att, query_image_ids=b_ids, is_training=False, top_k=top_k)
            ret_b["token_ids"] = ret_b["token_ids"].clamp(0, model.opt.vocab_size)
        seq, _ = model(fc, att, retrieval_batch=ret_b, opt={"beam_size": beam_size, "sample_n": 1}, mode="sample")
        for iid, cap in zip(b_ids, ConsistentCaptionEvaluator.decode_tokens(model.opt.vocab, seq)):
            preds.append({"image_id": int(iid), "caption": cap.strip()})
    return preds


def _save_experiment_history(cfg, key, payload):
    hist = {}
    if os.path.exists(cfg.TRAINING_HISTORY_JSON):
        try:
            with open(cfg.TRAINING_HISTORY_JSON, "r", encoding="utf-8") as f:
                hist = json.load(f)
        except Exception:
            hist = {}
    hist[key] = payload
    with open(cfg.TRAINING_HISTORY_JSON, "w", encoding="utf-8") as f:
        json.dump(hist, f, indent=2)


def train_model_ab(ctx: dict, run_name: str, exp_key: str, use_retrieval: bool, ab_mode: str):
    cfg = ctx["cfg"]
    setup_isolated_hatnet_repo(cfg)
    from captioning.data.dataloader import DataLoader

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    set_all_seeds(cfg.SEED)

    opt = build_opt_from_ctx(ctx)
    loader = DataLoader(opt)
    model = optimizer = scheduler = None

    try:
        opt.vocab_size = loader.dataset.get_vocab_size()
        opt.seq_length = loader.dataset.get_seq_length()
        opt.vocab = {str(k): v for k, v in loader.dataset.get_vocab().items()}

        ret_mem = ctx["retrieval_memory"]
        ret_mem.memory_token_ids.clamp_(0, opt.vocab_size)

        model = RetrievalEnhancedHATNet(
            opt, use_retrieval=use_retrieval, ab_mode=ab_mode,
            num_prompt_tokens=cfg.NUM_PROMPT_TOKENS, prompt_dropout=cfg.PROMPT_DROPOUT, vocab_idf=ctx["vocab_idf"]
        ).to(DEVICE)

        train_crit = MaskedLabelSmoothingLoss(smoothing=cfg.LABEL_SMOOTHING)
        optimizer = torch.optim.Adam(
            [p for p in model.parameters() if p.requires_grad],
            lr=cfg.LEARNING_RATE, betas=(0.9, 0.98), eps=1e-9
        )

        train_ids, val_ids = ctx["manifest"]["train_ids"], ctx["manifest"]["val_ids"]
        steps_per_ep = int(math.ceil(len(train_ids) / cfg.BATCH_SIZE))
        total_steps = steps_per_ep * cfg.MAX_EPOCHS
        warmup_steps = min(steps_per_ep * cfg.WARMUP_EPOCHS, max(1, total_steps // 5))
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            lambda s: (s + 1) / max(1, warmup_steps)
            if s < warmup_steps
            else max(0.05, 0.5 * (1.0 + math.cos(math.pi * (s - warmup_steps) / max(1, total_steps - warmup_steps))))
        )

        best_cider, best_ep, history = -1.0, -1, []
        best_ckpt = os.path.join(cfg.CHECKPOINT_DIR, f"{run_name}_best.pth")
        latest_ckpt = os.path.join(cfg.CHECKPOINT_DIR, f"{run_name}_latest.pth")

        print(f"\n=== [{cfg.ACTIVE_DATASET.upper()}] Training '{run_name}' ({cfg.MAX_EPOCHS} epochs) ===")
        for epoch in tqdm(range(1, cfg.MAX_EPOCHS + 1), desc=f"[{cfg.ACTIVE_DATASET}:{run_name}]"):
            t0 = time.time()
            model.train()
            loader.reset_iterator("train")
            tr_losses = []

            for _ in tqdm(range(steps_per_ep), desc=f"Ep {epoch:02d} Train", leave=False):
                optimizer.zero_grad(set_to_none=True)
                b = loader.get_batch("train")
                att = torch.as_tensor(b["att_feats"], device=DEVICE)
                lbl = torch.as_tensor(b["labels"], dtype=torch.long, device=DEVICE).clamp(0, opt.vocab_size)
                msk = torch.as_tensor(b["masks"], dtype=torch.float32, device=DEVICE)
                att_m = torch.as_tensor(b["att_masks"], device=DEVICE) if b.get("att_masks") is not None else None
                fc = torch.zeros(att.shape[0], 0, device=DEVICE)

                ret_b = None
                if model.use_retrieval:
                    ids = [int(inf["id"]) for inf in b["infos"]]
                    u_att = att if att.shape[0] == len(ids) else att[0::opt.seq_per_img]
                    ret_b = ret_mem.retrieve_batch(u_att, query_image_ids=ids, is_training=True, top_k=cfg.RETRIEVAL_TOP_K)
                    ret_b["token_ids"] = ret_b["token_ids"].clamp(0, opt.vocab_size)

                out = model(fc, att, lbl, att_m, retrieval_batch=ret_b, mode="forward")
                loss = train_crit(out, lbl[..., 1:], msk[..., 1:])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP_NORM)
                optimizer.step()
                scheduler.step()
                tr_losses.append(float(loss.item()))

            val_preds = generate_split_predictions(
                model, val_ids, ret_mem, cfg.FEATURES_DIR, cfg.RETRIEVAL_TOP_K, cfg.BEAM_SIZE, desc=f"Ep {epoch:02d} Val"
            )
            val_m = ConsistentCaptionEvaluator.evaluate_predictions(ctx["val_coco_json"], val_preds, set(val_ids))
            cider = float(val_m["CIDEr"])
            is_best = cider > best_cider
            if is_best:
                best_cider, best_ep = cider, epoch

            ckpt = {
                "dataset": cfg.ACTIVE_DATASET, "run_name": run_name, "epoch": epoch, "best_epoch": best_ep,
                "best_val_cider_raw": best_cider, "use_retrieval": use_retrieval, "ab_mode": ab_mode,
                "num_prompt_tokens": cfg.NUM_PROMPT_TOKENS, "prompt_dropout": cfg.PROMPT_DROPOUT,
                "model_state_dict": model.state_dict(),
                "opt_config": {k: v for k, v in vars(opt).items() if isinstance(v, (int, float, str, bool, list, dict)) or v is None},
            }
            _atomic_torch_save(ckpt, latest_ckpt)
            if is_best:
                _atomic_torch_save(ckpt, best_ckpt)
                with open(cfg.VAL_PREDS_JSON, "w", encoding="utf-8") as f:
                    json.dump(val_preds, f, indent=2)

            history.append({
                "epoch": epoch, "train_loss": float(np.mean(tr_losses)),
                "val_metrics_raw": val_m, "best_val_cider_raw": best_cider, "best_epoch": best_ep
            })
            print(
                f"[{cfg.ACTIVE_DATASET}:{run_name}] Ep {epoch:02d}/{cfg.MAX_EPOCHS} ({time.time()-t0:.1f}s) | "
                f"Loss: {np.mean(tr_losses):.4f} | Val B4: {val_m['Bleu_4']*100:.2f} | "
                f"Val CIDEr: {cider*100:.2f}{' *BEST*' if is_best else ''}"
            )
    finally:
        _close_dataloader_h5(loader)
        del loader, model, optimizer, scheduler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    res = {
        "run_name": run_name, "use_retrieval": use_retrieval, "ab_mode": ab_mode, "best_epoch": best_ep,
        "best_val_cider_raw": best_cider, "best_ckpt_path": best_ckpt, "latest_ckpt_path": latest_ckpt, "history": history
    }
    _save_experiment_history(cfg, exp_key, res)
    return res


def train_model_c_two_stage(ctx: dict):
    """Exact RSICD 2-Stage Warm-Start Trainer for Model C (Stage 1: Frozen Backbone, Stage 2: Joint Fine-Tune)."""
    cfg = ctx["cfg"]
    setup_isolated_hatnet_repo(cfg)
    from captioning.data.dataloader import DataLoader

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    set_all_seeds(cfg.SEED)

    opt = build_opt_from_ctx(ctx)
    loader = DataLoader(opt)
    model = optimizer = scheduler = None

    try:
        opt.vocab_size = loader.dataset.get_vocab_size()
        opt.seq_length = loader.dataset.get_seq_length()
        opt.vocab = {str(k): v for k, v in loader.dataset.get_vocab().items()}

        ret_mem = ctx["retrieval_memory"]
        ret_mem.memory_token_ids.clamp_(0, opt.vocab_size)

        model = RetrievalEnhancedHATNet(
            opt, use_retrieval=True, ab_mode="full_vipcap",
            num_prompt_tokens=cfg.NUM_PROMPT_TOKENS, prompt_dropout=cfg.PROMPT_DROPOUT, vocab_idf=ctx["vocab_idf"]
        ).to(DEVICE)

        model_b_ckpt = os.path.join(cfg.CHECKPOINT_DIR, "model_b_retrieval_only_best.pth")
        assert os.path.exists(model_b_ckpt), f"Run Model B for {cfg.ACTIVE_DATASET.upper()} before Model C! Missing: {model_b_ckpt}"
        try:
            ckpt_b = torch.load(model_b_ckpt, map_location=DEVICE, weights_only=True)
        except Exception:
            ckpt_b = torch.load(model_b_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt_b["model_state_dict"], strict=False)
        model.prompt_fusion.prompt_beta.data.zero_()
        print(f"[{cfg.ACTIVE_DATASET.upper()} Model C] Warm-started from Model B (Best Ep {ckpt_b['best_epoch']}).")

        train_crit = MaskedLabelSmoothingLoss(smoothing=cfg.LABEL_SMOOTHING)
        train_ids, val_ids = ctx["manifest"]["train_ids"], ctx["manifest"]["val_ids"]
        steps_per_ep = int(math.ceil(len(train_ids) / cfg.BATCH_SIZE))
        max_epochs = cfg.STAGE1_EPOCHS + cfg.STAGE2_EPOCHS

        run_name = "model_c_full_vipcap"
        best_ckpt = os.path.join(cfg.CHECKPOINT_DIR, f"{run_name}_best.pth")
        latest_ckpt = os.path.join(cfg.CHECKPOINT_DIR, f"{run_name}_latest.pth")

        val_preds0 = generate_split_predictions(model, val_ids, ret_mem, cfg.FEATURES_DIR, cfg.RETRIEVAL_TOP_K, cfg.BEAM_SIZE, desc="Ep 00 Anchor")
        val_m0 = ConsistentCaptionEvaluator.evaluate_predictions(ctx["val_coco_json"], val_preds0, set(val_ids))
        best_cider, best_ep = float(val_m0["CIDEr"]), 0
        print(f"[{cfg.ACTIVE_DATASET.upper()} Anchor] Ep 00 | Val B4: {val_m0['Bleu_4']*100:.2f} | Val CIDEr: {best_cider*100:.2f}")

        _atomic_torch_save({
            "dataset": cfg.ACTIVE_DATASET, "run_name": run_name, "epoch": 0, "best_epoch": 0,
            "best_val_cider_raw": best_cider, "use_retrieval": True, "ab_mode": "full_vipcap",
            "num_prompt_tokens": cfg.NUM_PROMPT_TOKENS, "prompt_dropout": cfg.PROMPT_DROPOUT,
            "model_state_dict": model.state_dict(),
            "opt_config": {k: v for k, v in vars(opt).items() if isinstance(v, (int, float, str, bool, list, dict)) or v is None},
        }, best_ckpt)

        vipcap_keys = ("prompt_mapper", "visual_prompt_attn", "prompt_gate_proj", "prompt_beta", "query_norm", "prompt_norm")
        history = []

        for epoch in tqdm(range(1, max_epochs + 1), desc=f"[{cfg.ACTIVE_DATASET}:{run_name}]"):
            t0 = time.time()
            if epoch == 1:
                print(f">>> Stage 1 (Ep 1-{cfg.STAGE1_EPOCHS}): Backbone frozen, training ViPCap modules (lr=3e-4)")
                for n, p in model.named_parameters():
                    p.requires_grad = any(k in n for k in vipcap_keys)
                optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=3e-4, betas=(0.9, 0.98), eps=1e-9)
                scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.STAGE1_EPOCHS * steps_per_ep, eta_min=5e-5)
            elif epoch == cfg.STAGE1_EPOCHS + 1:
                print(f">>> Stage 2 (Ep {cfg.STAGE1_EPOCHS+1}-{max_epochs}): Unfreezing all (Backbone 2e-5, ViPCap 1.5e-4)")
                vp, bp = [], []
                for n, p in model.named_parameters():
                    p.requires_grad = True
                    (vp if any(k in n for k in vipcap_keys) else bp).append(p)
                optimizer = torch.optim.Adam([{"params": bp, "lr": 2e-5}, {"params": vp, "lr": 1.5e-4}], betas=(0.9, 0.98), eps=1e-9)
                scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.STAGE2_EPOCHS * steps_per_ep, eta_min=1e-5)

            model.train()
            loader.reset_iterator("train")
            tr_losses = []
            for _ in tqdm(range(steps_per_ep), desc=f"Ep {epoch:02d} Train", leave=False):
                optimizer.zero_grad(set_to_none=True)
                b = loader.get_batch("train")
                att = torch.as_tensor(b["att_feats"], device=DEVICE)
                lbl = torch.as_tensor(b["labels"], dtype=torch.long, device=DEVICE).clamp(0, opt.vocab_size)
                msk = torch.as_tensor(b["masks"], dtype=torch.float32, device=DEVICE)
                att_m = torch.as_tensor(b["att_masks"], device=DEVICE) if b.get("att_masks") is not None else None
                fc = torch.zeros(att.shape[0], 0, device=DEVICE)
                ids = [int(inf["id"]) for inf in b["infos"]]
                u_att = att if att.shape[0] == len(ids) else att[0::opt.seq_per_img]
                ret_b = ret_mem.retrieve_batch(u_att, query_image_ids=ids, is_training=True, top_k=cfg.RETRIEVAL_TOP_K)
                ret_b["token_ids"] = ret_b["token_ids"].clamp(0, opt.vocab_size)

                out = model(fc, att, lbl, att_m, retrieval_batch=ret_b, mode="forward")
                loss = train_crit(out, lbl[..., 1:], msk[..., 1:])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP_NORM)
                optimizer.step()
                scheduler.step()
                tr_losses.append(float(loss.item()))

            val_preds = generate_split_predictions(
                model, val_ids, ret_mem, cfg.FEATURES_DIR, cfg.RETRIEVAL_TOP_K, cfg.BEAM_SIZE, desc=f"Ep {epoch:02d} Val"
            )
            val_m = ConsistentCaptionEvaluator.evaluate_predictions(ctx["val_coco_json"], val_preds, set(val_ids))
            cider = float(val_m["CIDEr"])
            is_best = cider >= best_cider
            if is_best:
                best_cider, best_ep = cider, epoch

            ckpt = {
                "dataset": cfg.ACTIVE_DATASET, "run_name": run_name, "epoch": epoch, "best_epoch": best_ep,
                "best_val_cider_raw": best_cider, "use_retrieval": True, "ab_mode": "full_vipcap",
                "num_prompt_tokens": cfg.NUM_PROMPT_TOKENS, "prompt_dropout": cfg.PROMPT_DROPOUT,
                "model_state_dict": model.state_dict(),
                "opt_config": {k: v for k, v in vars(opt).items() if isinstance(v, (int, float, str, bool, list, dict)) or v is None},
            }
            _atomic_torch_save(ckpt, latest_ckpt)
            if is_best:
                _atomic_torch_save(ckpt, best_ckpt)
                with open(cfg.VAL_PREDS_JSON, "w", encoding="utf-8") as f:
                    json.dump(val_preds, f, indent=2)

            history.append({
                "epoch": epoch, "train_loss": float(np.mean(tr_losses)),
                "val_metrics_raw": val_m, "best_val_cider_raw": best_cider, "best_epoch": best_ep
            })
            print(
                f"[{cfg.ACTIVE_DATASET}:{run_name}] Ep {epoch:02d}/{max_epochs} ({time.time()-t0:.1f}s) | "
                f"Loss: {np.mean(tr_losses):.4f} | Val B4: {val_m['Bleu_4']*100:.2f} | "
                f"Val CIDEr: {cider*100:.2f} | beta: {float(model.prompt_fusion.prompt_beta.item()):+.4f}"
                f"{' *BEST*' if is_best else ''}"
            )
    finally:
        _close_dataloader_h5(loader)
        del loader, model, optimizer, scheduler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    res = {
        "run_name": run_name, "use_retrieval": True, "ab_mode": "full_vipcap", "best_epoch": best_ep,
        "best_val_cider_raw": best_cider, "best_ckpt_path": best_ckpt, "latest_ckpt_path": latest_ckpt, "history": history
    }
    _save_experiment_history(cfg, "Model_C_Full_ViPCap", res)
    return res


def evaluate_dataset_on_test(ctx: dict) -> dict:
    """Evaluates Model A, B, and C best checkpoints on the dataset's test split and saves independent *_test_metrics.json."""
    cfg = ctx["cfg"]
    setup_isolated_hatnet_repo(cfg)
    ret_mem = ctx["retrieval_memory"]
    test_ids = ctx["manifest"]["test_ids"]

    model_specs = [
        ("Model_A_Baseline",       "model_a_baseline_best.pth"),
        ("Model_B_Retrieval_Only", "model_b_retrieval_only_best.pth"),
        ("Model_C_Full_ViPCap",    "model_c_full_vipcap_best.pth"),
    ]

    results = {}
    for label, ckpt_name in model_specs:
        ckpt_path = os.path.join(cfg.CHECKPOINT_DIR, ckpt_name)
        if not os.path.exists(ckpt_path):
            print(f"[SKIP] Missing checkpoint: {ckpt_path}")
            continue
        try:
            ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
        except Exception:
            ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

        opt_l = SimpleNamespace(**ckpt["opt_config"])
        model = RetrievalEnhancedHATNet(
            opt_l, use_retrieval=ckpt["use_retrieval"], ab_mode=ckpt["ab_mode"],
            num_prompt_tokens=ckpt["num_prompt_tokens"], prompt_dropout=ckpt.get("prompt_dropout", 0.1),
            vocab_idf=ctx["vocab_idf"]
        ).to(DEVICE)
        model.load_state_dict(ckpt["model_state_dict"], strict=True)
        model.eval()

        preds = generate_split_predictions(
            model, test_ids, ret_mem, cfg.FEATURES_DIR, cfg.RETRIEVAL_TOP_K, cfg.BEAM_SIZE,
            desc=f"[{cfg.ACTIVE_DATASET.upper()}] Test {label}"
        )
        t_metrics = ConsistentCaptionEvaluator.evaluate_predictions(ctx["test_coco_json"], preds, set(test_ids), try_spice=True)

        results[label] = {
            "best_epoch": int(ckpt["best_epoch"]),
            "best_val_cider_raw": float(ckpt["best_val_cider_raw"]),
            "test_metrics_raw": t_metrics,
        }
        pred_file = os.path.join(cfg.RESULTS_DIR, f"{cfg.ACTIVE_DATASET}_test_preds_{label.lower()}.json")
        with open(pred_file, "w", encoding="utf-8") as f:
            json.dump(preds, f, indent=2)
        if label == "Model_C_Full_ViPCap":
            with open(cfg.TEST_PREDS_JSON, "w", encoding="utf-8") as f:
                json.dump(preds, f, indent=2)

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    with open(cfg.TEST_METRICS_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    metric_keys = ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L", "CIDEr"]
    header = f"{cfg.ACTIVE_DATASET.upper() + ' Model Configuration':28s} | {'BestEp':>6s} | " + " | ".join(f"{m:>7s}" for m in metric_keys)
    print("\n" + "=" * len(header))
    print(header)
    print("-" * len(header))
    base = results.get("Model_A_Baseline", {}).get("test_metrics_raw", {})
    for label, info in results.items():
        tm = info["test_metrics_raw"]
        print(f"{label:28s} | {info['best_epoch']:6d} | " + " | ".join(f"{tm[m]*100:7.2f}" for m in metric_keys))
        if label != "Model_A_Baseline" and base:
            print(f"{'  (Delta vs. Model A)':28s} | {'':6s} | " + " | ".join(f"{(tm[m]-base[m])*100:+7.2f}" for m in metric_keys))
    print("=" * len(header))
    print(f"[SAVED] Independent Test Metrics -> {cfg.TEST_METRICS_JSON}")

    with open(cfg.RUN_SUMMARY_TXT, "w", encoding="utf-8") as f:
        f.write(f"Dataset: {cfg.ACTIVE_DATASET.upper()} | Timestamp: {datetime.datetime.now().isoformat()}\n")
        f.write(json.dumps(results, indent=2))
    return results

print("[CELL 6 READY] Debugged Training & Evaluation engine loaded.")

[CELL 6 READY] Debugged Training & Evaluation engine loaded.


In [19]:
# CELL 7A: Prepare UCM-Captions (1,680 Train / 210 Val / 210 Test)
ucm_cfg = build_dataset_config("ucm")
ucm_ctx = prepare_dataset_context(ucm_cfg)
extract_and_verify_features(ucm_ctx)
ucm_ctx["retrieval_memory"] = TrainingOnlyRetrievalMemory(ucm_ctx)
print(f"[UCM READY FOR TRAINING] Retrieval Memory: {len(ucm_ctx['retrieval_memory'].train_ids)} images")

[UCM CONFIG READY] Workspace: C:\Users\LENOVO\Desktop\vansh_ucm | Split: 1680/210/210 (Total 2100)
[UCM HF LOADED] 2100 rows | Columns: ['image', 'file_name', 'caption_1', 'caption_2', 'caption_3', 'caption_4', 'caption_5']
[UCM EXPORTED] 2100 records | Caption cols: ['caption_1', 'caption_2', 'caption_3', 'caption_4', 'caption_5'] | Sample[0]: 'lots of boats docked at the harbor and the water is deep blue'
[UCM] Reusing verified stratified split manifest.
[UCM READY] Train/Val/Test = 1680/210/210 | Vocab = 306
[UCM] Using GeoRSCLIP checkpoint: C:\Users\LENOVO\Desktop\vansh_ucm\ckpt\RS5M_ViT-H-14.pt
[UCM CACHE] 1250 valid cached | 850 to extract.


Extracting UCM GeoRSCLIP:   0%|          | 0/54 [00:00<?, ?it/s]

[UCM] All 2100 feature files verified!
[UCM READY FOR TRAINING] Retrieval Memory: 1680 images


In [24]:
# CELL 7B: UCM Model A (Baseline)
ucm_res_a = train_model_ab(ucm_ctx, "model_a_baseline", "Model_A_Baseline", use_retrieval=False, ab_mode="baseline")

DataLoader loading json file:  C:\Users\LENOVO\Desktop\vansh_ucm\data\ucm_info.json
vocab size is  306
DataLoader loading h5 file:   C:\Users\LENOVO\Desktop\vansh_ucm\features  C:\Users\LENOVO\Desktop\vansh_ucm\data\ucm_label.h5
max sequence length in data is 20
read 2100 image features
assigned 1680 images to split train
assigned 210 images to split val
assigned 210 images to split test


C:\Users\LENOVO\anaconda3\Lib\site-packages\torch\nn\modules\linear.py:124: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))



=== [UCM] Training 'model_a_baseline' (18 epochs) ===


[ucm:model_a_baseline]:   0%|          | 0/18 [00:00<?, ?it/s]

Ep 01 Train:   0%|          | 0/105 [00:00<?, ?it/s]

Ep 01 Val:   0%|          | 0/7 [00:00<?, ?it/s]

... (45 outputs left out)

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
{'testlen': 2017, 'reflen': 2008, 'guess': [2017, 1807, 1597, 1387], 'correct': [1877, 1550, 1283, 1055]}
ratio: 1.0044820717126473
[ucm:model_a_baseline] Ep 17/18 (133.6s) | Loss: 0.6921 | Val B4: 83.57 | Val CIDEr: 428.52


Ep 18 Train:   0%|          | 0/105 [00:00<?, ?it/s]

Ep 18 Val:   0%|          | 0/7 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 2053, 'reflen': 2040, 'guess': [2053, 1843, 1633, 1423], 'correct': [1904, 1573, 1300, 1070]}
ratio: 1.0063725490191144
[ucm:model_a_baseline] Ep 18/18 (137.5s) | Loss: 0.6891 | Val B4: 82.97 | Val CIDEr: 421.17


In [25]:
# CELL 7C: UCM Model B (Retrieval-Only)
ucm_res_b = train_model_ab(ucm_ctx, "model_b_retrieval_only", "Model_B_Retrieval_Only", use_retrieval=True, ab_mode="retrieval_only")

DataLoader loading json file:  C:\Users\LENOVO\Desktop\vansh_ucm\data\ucm_info.json
vocab size is  306
DataLoader loading h5 file:   C:\Users\LENOVO\Desktop\vansh_ucm\features  C:\Users\LENOVO\Desktop\vansh_ucm\data\ucm_label.h5
max sequence length in data is 20
read 2100 image features
assigned 1680 images to split train
assigned 210 images to split val
assigned 210 images to split test

=== [UCM] Training 'model_b_retrieval_only' (18 epochs) ===


[ucm:model_b_retrieval_only]:   0%|          | 0/18 [00:00<?, ?it/s]

Ep 01 Train:   0%|          | 0/105 [00:00<?, ?it/s]

Ep 01 Val:   0%|          | 0/7 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 2041, 'reflen': 2055, 'guess': [2041, 1831, 1621, 1411], 'correct': [886, 424, 260, 183]}
ratio: 0.9931873479313902
[ucm:model_b_retrieval_only] Ep 01/18 (150.6s) | Loss: 3.5368 | Val B4: 21.24 | Val CIDEr: 60.71 *BEST*


Ep 02 Train:   0%|          | 0/105 [00:00<?, ?it/s]

... (43 outputs left out)

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
{'testlen': 2058, 'reflen': 2058, 'guess': [2058, 1848, 1638, 1428], 'correct': [1938, 1629, 1365, 1133]}
ratio: 0.999999999999514
[ucm:model_b_retrieval_only] Ep 16/18 (133.7s) | Loss: 0.6894 | Val B4: 86.07 | Val CIDEr: 454.58


Ep 17 Train:   0%|          | 0/105 [00:00<?, ?it/s]

Ep 18 Val:   0%|          | 0/7 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 2079, 'reflen': 2074, 'guess': [2079, 1869, 1659, 1449], 'correct': [1972, 1670, 1405, 1173]}
ratio: 1.0024108003852448
[ucm:model_b_retrieval_only] Ep 18/18 (132.9s) | Loss: 0.6842 | Val B4: 87.31 | Val CIDEr: 460.35 *BEST*


In [26]:
# CELL 7D: UCM Model C (Full ViPCap, 2-Stage Warm-Start from UCM Model B)
ucm_res_c = train_model_c_two_stage(ucm_ctx)

DataLoader loading json file:  C:\Users\LENOVO\Desktop\vansh_ucm\data\ucm_info.json
vocab size is  306
DataLoader loading h5 file:   C:\Users\LENOVO\Desktop\vansh_ucm\features  C:\Users\LENOVO\Desktop\vansh_ucm\data\ucm_label.h5
max sequence length in data is 20
read 2100 image features
assigned 1680 images to split train
assigned 210 images to split val
assigned 210 images to split test
[UCM Model C] Warm-started from Model B (Best Ep 18).


Ep 00 Anchor:   0%|          | 0/7 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
{'testlen': 2079, 'reflen': 2074, 'guess': [2079, 1869, 1659, 1449], 'correct': [1972, 1670, 1405, 1173]}
ratio: 1.0024108003852448
[UCM Anchor] Ep 00 | Val B4: 87.31 | Val CIDEr: 460.35


[ucm:model_c_full_vipcap]:   0%|          | 0/10 [00:00<?, ?it/s]

>>> Stage 1 (Ep 1-3): Backbone frozen, training ViPCap modules (lr=3e-4)


Ep 01 Train:   0%|          | 0/105 [00:00<?, ?it/s]

... (25 outputs left out)

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 2108, 'reflen': 2105, 'guess': [2108, 1898, 1688, 1478], 'correct': [1987, 1681, 1417, 1182]}
ratio: 1.0014251781467927
[ucm:model_c_full_vipcap] Ep 09/10 (141.9s) | Loss: 0.6745 | Val B4: 86.52 | Val CIDEr: 457.22 | beta: -0.0129


Ep 10 Train:   0%|          | 0/105 [00:00<?, ?it/s]

Ep 10 Val:   0%|          | 0/7 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
{'testlen': 2115, 'reflen': 2096, 'guess': [2115, 1905, 1695, 1485], 'correct': [1991, 1680, 1413, 1177]}
ratio: 1.0090648854957018
[ucm:model_c_full_vipcap] Ep 10/10 (139.4s) | Loss: 0.6738 | Val B4: 86.06 | Val CIDEr: 453.94 | beta: -0.0129


In [27]:
# CELL 7E: Final Independent UCM Test Evaluation -> saves ucm_test_metrics.json
ucm_test_metrics = evaluate_dataset_on_test(ucm_ctx)

[UCM] Test Model_A_Baseline:   0%|          | 0/7 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 2027, 'reflen': 2030, 'guess': [2027, 1817, 1607, 1397], 'correct': [1848, 1521, 1247, 1022]}
ratio: 0.9985221674871928


[UCM] Test Model_B_Retrieval_Only:   0%|          | 0/7 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
{'testlen': 2105, 'reflen': 2111, 'guess': [2105, 1895, 1685, 1475], 'correct': [1967, 1674, 1406, 1172]}
ratio: 0.9971577451440089


[UCM] Test Model_C_Full_ViPCap:   0%|          | 0/7 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
{'testlen': 2102, 'reflen': 2111, 'guess': [2102, 1892, 1682, 1472], 'correct': [1958, 1664, 1398, 1166]}
ratio: 0.9957366177162502

UCM Model Configuration      | BestEp |  Bleu_1 |  Bleu_2 |  Bleu_3 |  Bleu_4 |  METEOR | ROUGE_L |   CIDEr
-----------------------------------------------------------------------------------------------------------
Model_A_Baseline             |     16 |   91.03 |   87.23 |   83.85 |   81.01 |   87.58 |   88.26 |  423.64
Model_B_Retrieval_Only       |     18 |   93.18 |   90.60 |   88.06 |   85.77 |   90.09 |   90.54 |  447.80
  (Delta vs. Model A)        |        |   +2.14 |   +3.37 |   +4.21 |   +4.76 |   +2.51 |   +2.28 |  +24.16
Model_C_Full_ViPCap          |      3 |   92.75 |   90.13 |   87.60 |   85.33 |   89.54 |   90.00 |  448.13
  (Delta vs. Model A)        |        |   +1.72 |   +2.89 |   +3.75 |   +4.32 |   +1.96 |   +1.74 |  +24.48
[SAVED] Independent Test Met

In [7]:
# CELL 8A: Prepare NWPU-Captions (Stratified 4,500: 3,600 Train / 450 Val / 450 Test)
# (Optional: set total_images=8000, train_count=7000, val_count=500, test_count=500 if you have ~30 GB free SSD space)
nwpu_cfg = build_dataset_config("nwpu", total_images=4500, train_count=3600, val_count=450, test_count=450)
nwpu_ctx = prepare_dataset_context(nwpu_cfg)
extract_and_verify_features(nwpu_ctx)
nwpu_ctx["retrieval_memory"] = TrainingOnlyRetrievalMemory(nwpu_ctx)
print(f"[NWPU READY FOR TRAINING] Retrieval Memory: {len(nwpu_ctx['retrieval_memory'].train_ids)} images")

[NWPU CONFIG READY] Workspace: C:\Users\LENOVO\Desktop\vansh_nwpu | Split: 3600/450/450 (Total 4500)


[NWPU HF LOADED] 31500 rows | Columns: ['image', 'file_name', 'caption_1', 'caption_2', 'caption_3', 'caption_4', 'caption_5']
[NWPU EXPORTED] 31500 records | Sample[0]: 'the cross shaped palace is surrounded by some other buildings'
[NWPU] Reusing verified stratified split manifest.
[NWPU READY] Train/Val/Test = 3600/450/450 | Vocab = 1078
[NWPU] Using GeoRSCLIP checkpoint: C:\Users\LENOVO\Desktop\vansh_ucm\ckpt\RS5M_ViT-H-14.pt
[NWPU CACHE] 4500 valid cached | 0 to extract.


Extracting NWPU GeoRSCLIP: 0it [00:00, ?it/s]

[NWPU] All 4500 feature files verified!
[NWPU READY FOR TRAINING] Retrieval Memory: 3600 images


In [8]:
# CELL 8B: NWPU Model A (Baseline)
nwpu_res_a = train_model_ab(nwpu_ctx, "model_a_baseline", "Model_A_Baseline", use_retrieval=False, ab_mode="baseline")

DataLoader loading json file:  C:\Users\LENOVO\Desktop\vansh_nwpu\data\nwpu_info.json
vocab size is  1078
DataLoader loading h5 file:   C:\Users\LENOVO\Desktop\vansh_nwpu\features  C:\Users\LENOVO\Desktop\vansh_nwpu\data\nwpu_label.h5
max sequence length in data is 25
read 4500 image features
assigned 3600 images to split train
assigned 450 images to split val
assigned 450 images to split test


C:\Users\LENOVO\anaconda3\Lib\site-packages\torch\nn\modules\linear.py:124: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))



=== [NWPU] Training 'model_a_baseline' (18 epochs) ===


[nwpu:model_a_baseline]:   0%|          | 0/18 [00:00<?, ?it/s]

Ep 01 Train:   0%|          | 0/225 [00:00<?, ?it/s]

Ep 01 Val:   0%|          | 0/15 [00:00<?, ?it/s]

... (48 outputs left out)

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 4562, 'reflen': 4542, 'guess': [4562, 4112, 3662, 3212], 'correct': [3991, 2956, 2227, 1701]}
ratio: 1.0044033465431517
[nwpu:model_a_baseline] Ep 17/18 (312.3s) | Loss: 0.8777 | Val B4: 67.09 | Val CIDEr: 198.32 *BEST*


Ep 18 Train:   0%|          | 0/225 [00:00<?, ?it/s]

Ep 18 Val:   0%|          | 0/15 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 4556, 'reflen': 4508, 'guess': [4556, 4106, 3656, 3206], 'correct': [4006, 2996, 2279, 1761]}
ratio: 1.0106477373555875
[nwpu:model_a_baseline] Ep 18/18 (324.3s) | Loss: 0.8741 | Val B4: 68.46 | Val CIDEr: 197.16


In [9]:
# CELL 8C: NWPU Model B (Retrieval-Only)
nwpu_res_b = train_model_ab(nwpu_ctx, "model_b_retrieval_only", "Model_B_Retrieval_Only", use_retrieval=True, ab_mode="retrieval_only")

DataLoader loading json file:  C:\Users\LENOVO\Desktop\vansh_nwpu\data\nwpu_info.json
vocab size is  1078
DataLoader loading h5 file:   C:\Users\LENOVO\Desktop\vansh_nwpu\features  C:\Users\LENOVO\Desktop\vansh_nwpu\data\nwpu_label.h5
max sequence length in data is 25
read 4500 image features
assigned 3600 images to split train
assigned 450 images to split val
assigned 450 images to split test

=== [NWPU] Training 'model_b_retrieval_only' (18 epochs) ===


[nwpu:model_b_retrieval_only]:   0%|          | 0/18 [00:00<?, ?it/s]

Ep 01 Train:   0%|          | 0/225 [00:00<?, ?it/s]

Ep 01 Val:   0%|          | 0/15 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 3749, 'reflen': 3725, 'guess': [3749, 3299, 2849, 2399], 'correct': [2558, 1269, 624, 298]}
ratio: 1.006442953019864
[nwpu:model_b_retrieval_only] Ep 01/18 (352.4s) | Loss: 4.2674 | Val B4: 29.07 | Val CIDEr: 46.98 *BEST*


Ep 02 Train:   0%|          | 0/225 [00:00<?, ?it/s]

... (44 outputs left out)

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 4652, 'reflen': 4626, 'guess': [4652, 4202, 3752, 3302], 'correct': [4097, 3075, 2336, 1822]}
ratio: 1.0056204063983991
[nwpu:model_b_retrieval_only] Ep 17/18 (317.3s) | Loss: 0.8760 | Val B4: 68.60 | Val CIDEr: 202.68


Ep 18 Train:   0%|          | 0/225 [00:00<?, ?it/s]

Ep 18 Val:   0%|          | 0/15 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 4679, 'reflen': 4666, 'guess': [4679, 4229, 3779, 3329], 'correct': [4149, 3120, 2387, 1867]}
ratio: 1.0027861123015425
[nwpu:model_b_retrieval_only] Ep 18/18 (317.2s) | Loss: 0.8724 | Val B4: 69.38 | Val CIDEr: 202.76


In [10]:
# CELL 8D: NWPU Model C (Full ViPCap, 2-Stage Warm-Start from NWPU Model B)
nwpu_res_c = train_model_c_two_stage(nwpu_ctx)

DataLoader loading json file:  C:\Users\LENOVO\Desktop\vansh_nwpu\data\nwpu_info.json
vocab size is  1078
DataLoader loading h5 file:   C:\Users\LENOVO\Desktop\vansh_nwpu\features  C:\Users\LENOVO\Desktop\vansh_nwpu\data\nwpu_label.h5
max sequence length in data is 25
read 4500 image features
assigned 3600 images to split train
assigned 450 images to split val
assigned 450 images to split test
[NWPU Model C] Warm-started from Model B (Best Ep 15).


Ep 00 Anchor:   0%|          | 0/15 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 4557, 'reflen': 4528, 'guess': [4557, 4107, 3657, 3207], 'correct': [4059, 3089, 2371, 1847]}
ratio: 1.0064045936393535
[NWPU Anchor] Ep 00 | Val B4: 70.72 | Val CIDEr: 207.06


[nwpu:model_c_full_vipcap]:   0%|          | 0/10 [00:00<?, ?it/s]

>>> Stage 1 (Ep 1-3): Backbone frozen, training ViPCap modules (lr=3e-4)


Ep 01 Train:   0%|          | 0/225 [00:00<?, ?it/s]

... (25 outputs left out)

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 4631, 'reflen': 4572, 'guess': [4631, 4181, 3731, 3281], 'correct': [4096, 3070, 2341, 1824]}
ratio: 1.0129046369201633
[nwpu:model_c_full_vipcap] Ep 09/10 (329.9s) | Loss: 0.8607 | Val B4: 68.99 | Val CIDEr: 202.92 | beta: +0.0327


Ep 10 Train:   0%|          | 0/225 [00:00<?, ?it/s]

Ep 10 Val:   0%|          | 0/15 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 4620, 'reflen': 4594, 'guess': [4620, 4170, 3720, 3270], 'correct': [4100, 3081, 2343, 1814]}
ratio: 1.0056595559423147
[nwpu:model_c_full_vipcap] Ep 10/10 (337.8s) | Loss: 0.8576 | Val B4: 69.18 | Val CIDEr: 203.70 | beta: +0.0329


In [11]:
# CELL 8E: Final Independent NWPU Test Evaluation -> saves nwpu_test_metrics.json
nwpu_test_metrics = evaluate_dataset_on_test(nwpu_ctx)

[NWPU] Test Model_A_Baseline:   0%|          | 0/15 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 4593, 'reflen': 4522, 'guess': [4593, 4143, 3693, 3243], 'correct': [4090, 3073, 2320, 1795]}
ratio: 1.0157010172487801


[NWPU] Test Model_B_Retrieval_Only:   0%|          | 0/15 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 4505, 'reflen': 4433, 'guess': [4505, 4055, 3605, 3155], 'correct': [3970, 2992, 2249, 1698]}
ratio: 1.0162418226932062


[NWPU] Test Model_C_Full_ViPCap:   0%|          | 0/15 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
{'testlen': 4622, 'reflen': 4547, 'guess': [4622, 4172, 3722, 3272], 'correct': [4030, 3016, 2269, 1732]}
ratio: 1.0164943919065281

NWPU Model Configuration     | BestEp |  Bleu_1 |  Bleu_2 |  Bleu_3 |  Bleu_4 |  METEOR | ROUGE_L |   CIDEr
-----------------------------------------------------------------------------------------------------------
Model_A_Baseline             |     17 |   89.05 |   81.27 |   74.59 |   69.23 |   78.00 |   78.21 |  204.50
Model_B_Retrieval_Only       |     15 |   88.12 |   80.64 |   74.03 |   68.36 |   77.84 |   78.05 |  205.26
  (Delta vs. Model A)        |        |   -0.92 |   -0.63 |   -0.56 |   -0.87 |   -0.16 |   -0.17 |   +0.77
Model_C_Full_ViPCap          |      4 |   87.19 |   79.39 |   72.70 |   67.16 |   76.57 |   76.86 |  199.12
  (Delta vs. Model A)        |        |   -1.86 |   -1.88 |   -1.89 |   -2.07 |   -1.43 |   -1.36 |   -5.37
[SAVED] Independent Test Met

In [12]:
# CELL 9: Unified Cross-Dataset Benchmark Table (RSICD [Read-Only] + UCM + NWPU)
desktop = os.path.join(os.path.expanduser("~"), "Desktop")
sources = [
    ("RSICD (8k)", [
        os.path.join(desktop, "vansh", "8k", "test_metrics.json"),
        os.path.join(desktop, "vansh_2", "test_metrics.json"),
    ]),
    ("UCM-Captions", [
        os.path.join(desktop, "vansh_ucm", "ucm_test_metrics.json"),
    ]),
    ("NWPU-Captions", [
        os.path.join(desktop, "vansh_nwpu", "nwpu_test_metrics.json"),
    ]),
]

metric_keys = ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L", "CIDEr"]
header = f"{'Dataset':14s} | {'Model':24s} | {'Ep':>3s} | " + " | ".join(f"{m:>7s}" for m in metric_keys)
print("\n" + "=" * len(header))
print(header)
print("=" * len(header))

for ds_label, paths in sources:
    found = next((p for p in paths if os.path.exists(p)), None)
    if not found:
        print(f"{ds_label:14s} | [Not yet generated]")
        print("-" * len(header))
        continue
    with open(found, "r", encoding="utf-8") as f:
        data = json.load(f)
    base = data.get("Model_A_Baseline", {}).get("test_metrics_raw", {})
    for m_key in ["Model_A_Baseline", "Model_B_Retrieval_Only", "Model_C_Full_ViPCap"]:
        if m_key not in data:
            continue
        tm = data[m_key]["test_metrics_raw"]
        ep = data[m_key].get("best_epoch", 0)
        print(f"{ds_label:14s} | {m_key:24s} | {ep:3d} | " + " | ".join(f"{tm[k]*100:7.2f}" for k in metric_keys))
        if m_key != "Model_A_Baseline" and base:
            print(f"{'':14s} | {'  (Delta vs A)':24s} | {'':3s} | " + " | ".join(f"{(tm[k]-base[k])*100:+7.2f}" for k in metric_keys))
    print("-" * len(header))


Dataset        | Model                    |  Ep |  Bleu_1 |  Bleu_2 |  Bleu_3 |  Bleu_4 |  METEOR | ROUGE_L |   CIDEr
RSICD (8k)     | Model_A_Baseline         |   7 |   72.91 |   61.30 |   52.77 |   46.40 |   61.98 |   62.57 |  254.47
RSICD (8k)     | Model_B_Retrieval_Only   |   7 |   72.52 |   61.41 |   53.09 |   46.81 |   62.20 |   63.30 |  256.34
               |   (Delta vs A)           |     |   -0.40 |   +0.11 |   +0.32 |   +0.42 |   +0.21 |   +0.73 |   +1.87
RSICD (8k)     | Model_C_Full_ViPCap      |   4 |   73.65 |   62.48 |   54.21 |   47.93 |   63.50 |   63.60 |  259.46
               |   (Delta vs A)           |     |   +0.73 |   +1.18 |   +1.44 |   +1.53 |   +1.52 |   +1.03 |   +4.99
---------------------------------------------------------------------------------------------------------------------
UCM-Captions   | Model_A_Baseline         |  16 |   91.03 |   87.23 |   83.85 |   81.01 |   87.58 |   88.26 |  423.64
UCM-Captions   | Model_B_Retrieval_Only   |  18 |   93.